In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


CELL 1 — INITIALIZE ADAPTIVE BILATERAL FUSION ENVIRONMENT

This notebook implements the Adaptive Bilateral Fusion stage of
QCDP-BiFormer.

The Cross-Eye Bilateral Attention stage has already been completed
and is treated as FROZEN.

Selected Cross-Eye model:

    Experiment 2

Final Cross-Eye representations:

    Training:
        2141 bilateral patients
        Right features  : [2141, 768]
        Left features   : [2141, 768]
        Patient features: [2141, 768]

    Validation:
        438 bilateral patients
        Right features  : [438, 768]
        Left features   : [438, 768]
        Patient features: [438, 768]

Cross-Eye attention tensors are also preserved.

The purpose of Adaptive Bilateral Fusion is NOT to repeat
Cross-Eye reasoning.

Instead, it learns how much each eye should contribute to the
final patient-level representation using:

    • Cross-Eye right-eye representation
    • Cross-Eye left-eye representation
    • Right-eye image quality
    • Left-eye image quality
    • Right-eye disease relevance
    • Left-eye disease relevance

The patient-level Cross-Eye representation is retained as an
additional reference rather than being blindly concatenated with
the two eye representations.

IMPORTANT METHODOLOGICAL RULES

    • Cross-Eye parameters remain frozen.
    • Stage-1 features remain frozen.
    • Disease prototypes remain frozen.
    • Training and validation remain strictly separated.
    • Patient ordering must remain unchanged.
    • No random reordering is allowed during final memory creation.
    • No information from validation is used to fit training components.
    • Adaptive Fusion will be the only newly trained module in this stage.

This cell only initializes the environment and loads previously
verified artifacts.

No model training is performed here.

In [3]:
# ============================================================
# CELL 1 — INITIALIZE ADAPTIVE BILATERAL FUSION ENVIRONMENT
# ============================================================

# ============================================================
# 1. GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")


# ============================================================
# 2. STANDARD LIBRARIES
# ============================================================

import os
import glob
import json
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    f1_score,
    roc_auc_score
)

from IPython.display import display


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 4. DEVICE
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 80)
print("ADAPTIVE BILATERAL FUSION — ENVIRONMENT INITIALIZATION")
print("=" * 80)

print("Device:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

else:

    print(
        "WARNING: CUDA is not available."
    )


# ============================================================
# 5. ROOT DIRECTORY
# ============================================================

ROOT = (
    "/content/drive/My Drive/"
    "Eye Disease/Dataset"
)

assert os.path.exists(ROOT), (
    f"Dataset directory not found:\n{ROOT}"
)

print("\nDataset root:")
print(ROOT)


# ============================================================
# 6. CORE CONSTANTS
# ============================================================

FEATURE_DIM = 768
NUM_CLASSES = 8

LABEL_COLUMNS = [
    "N",
    "D",
    "G",
    "C",
    "A",
    "H",
    "M",
    "O"
]

EXPECTED_TRAIN_PATIENTS = 2141
EXPECTED_VAL_PATIENTS = 438

print("\nConfiguration:")
print("Feature dimension:", FEATURE_DIM)
print("Number of diseases:", NUM_CLASSES)
print("Disease labels:", LABEL_COLUMNS)
print("Expected training patients:", EXPECTED_TRAIN_PATIENTS)
print("Expected validation patients:", EXPECTED_VAL_PATIENTS)


# ============================================================
# 7. FROZEN CROSS-EYE ARTIFACTS
# ============================================================

FINAL_TRAIN_CROSS_EYE_PATH = os.path.join(
    ROOT,
    "stage2_train_cross_eye_features_final.pt"
)

FINAL_VAL_CROSS_EYE_PATH = os.path.join(
    ROOT,
    "stage2_val_cross_eye_features_final.pt"
)


assert os.path.exists(
    FINAL_TRAIN_CROSS_EYE_PATH
), (
    "Final training Cross-Eye artifact not found:\n"
    f"{FINAL_TRAIN_CROSS_EYE_PATH}"
)

assert os.path.exists(
    FINAL_VAL_CROSS_EYE_PATH
), (
    "Final validation Cross-Eye artifact not found:\n"
    f"{FINAL_VAL_CROSS_EYE_PATH}"
)


# ============================================================
# 8. LOAD FINAL CROSS-EYE REPRESENTATIONS
# ============================================================

cross_eye_train = torch.load(
    FINAL_TRAIN_CROSS_EYE_PATH,
    map_location="cpu",
    weights_only=False
)

cross_eye_val = torch.load(
    FINAL_VAL_CROSS_EYE_PATH,
    map_location="cpu",
    weights_only=False
)


# ============================================================
# 9. VERIFY CROSS-EYE ARTIFACT STRUCTURE
# ============================================================

required_cross_eye_keys = [
    "right_features",
    "left_features",
    "patient_features",
    "right_attention",
    "left_attention",
    "patient_ids"
]

for key in required_cross_eye_keys:

    assert key in cross_eye_train, (
        f"Missing training Cross-Eye key: {key}"
    )

    assert key in cross_eye_val, (
        f"Missing validation Cross-Eye key: {key}"
    )


# ============================================================
# 10. EXTRACT CROSS-EYE TENSORS
# ============================================================

train_cross_right = (
    cross_eye_train[
        "right_features"
    ].float()
)

train_cross_left = (
    cross_eye_train[
        "left_features"
    ].float()
)

train_cross_patient = (
    cross_eye_train[
        "patient_features"
    ].float()
)

train_cross_right_attention = (
    cross_eye_train[
        "right_attention"
    ].float()
)

train_cross_left_attention = (
    cross_eye_train[
        "left_attention"
    ].float()
)

train_patient_ids = (
    cross_eye_train[
        "patient_ids"
    ]
)


val_cross_right = (
    cross_eye_val[
        "right_features"
    ].float()
)

val_cross_left = (
    cross_eye_val[
        "left_features"
    ].float()
)

val_cross_patient = (
    cross_eye_val[
        "patient_features"
    ].float()
)

val_cross_right_attention = (
    cross_eye_val[
        "right_attention"
    ].float()
)

val_cross_left_attention = (
    cross_eye_val[
        "left_attention"
    ].float()
)

val_patient_ids = (
    cross_eye_val[
        "patient_ids"
    ]
)


# ============================================================
# 11. CROSS-EYE SHAPE VERIFICATION
# ============================================================

assert train_cross_right.shape == (
    EXPECTED_TRAIN_PATIENTS,
    FEATURE_DIM
)

assert train_cross_left.shape == (
    EXPECTED_TRAIN_PATIENTS,
    FEATURE_DIM
)

assert train_cross_patient.shape == (
    EXPECTED_TRAIN_PATIENTS,
    FEATURE_DIM
)

assert val_cross_right.shape == (
    EXPECTED_VAL_PATIENTS,
    FEATURE_DIM
)

assert val_cross_left.shape == (
    EXPECTED_VAL_PATIENTS,
    FEATURE_DIM
)

assert val_cross_patient.shape == (
    EXPECTED_VAL_PATIENTS,
    FEATURE_DIM
)

assert train_cross_right_attention.shape == (
    EXPECTED_TRAIN_PATIENTS,
    4,
    8,
    8
)

assert train_cross_left_attention.shape == (
    EXPECTED_TRAIN_PATIENTS,
    4,
    8,
    8
)

assert val_cross_right_attention.shape == (
    EXPECTED_VAL_PATIENTS,
    4,
    8,
    8
)

assert val_cross_left_attention.shape == (
    EXPECTED_VAL_PATIENTS,
    4,
    8,
    8
)


# ============================================================
# 12. LOAD STAGE-1 QUALITY SCORES
# ============================================================

QUALITY_SCORE_PATH = os.path.join(
    ROOT,
    "stage1_quality_scores.csv"
)

assert os.path.exists(
    QUALITY_SCORE_PATH
), (
    "Stage-1 quality-score file not found:\n"
    f"{QUALITY_SCORE_PATH}"
)

stage1_quality_df = pd.read_csv(
    QUALITY_SCORE_PATH
)


# ============================================================
# 13. VERIFY QUALITY SCORE STRUCTURE
# ============================================================

required_quality_columns = [
    "filename",
    "patient_id",
    "eye",
    "quality_score"
]

missing_quality_columns = [
    col
    for col in required_quality_columns
    if col not in stage1_quality_df.columns
]

assert not missing_quality_columns, (
    "Missing quality columns: "
    + str(missing_quality_columns)
)

assert len(stage1_quality_df) == 4491

assert stage1_quality_df[
    "quality_score"
].notna().all()

assert np.isfinite(
    stage1_quality_df[
        "quality_score"
    ].to_numpy()
).all()

assert (
    stage1_quality_df[
        "quality_score"
    ].between(0, 1).all()
)


# ============================================================
# 14. LOAD ORIGINAL STAGE-1 LEARNED FEATURES
# ============================================================

TRAIN_FEATURES_PATH = os.path.join(
    ROOT,
    "train_features.pt"
)

assert os.path.exists(
    TRAIN_FEATURES_PATH
), (
    "Stage-1 learned feature file not found:\n"
    f"{TRAIN_FEATURES_PATH}"
)

stage1_train_features_artifact = torch.load(
    TRAIN_FEATURES_PATH,
    map_location="cpu",
    weights_only=False
)


# ============================================================
# 15. INSPECT STAGE-1 FEATURE ARTIFACT
# ============================================================

print("\nStage-1 feature artifact type:")

print(
    type(
        stage1_train_features_artifact
    )
)

if isinstance(
    stage1_train_features_artifact,
    dict
):

    print(
        "Available keys:",
        list(
            stage1_train_features_artifact.keys()
        )
    )

else:

    print(
        "Artifact is not a dictionary."
    )


# ============================================================
# 16. LOAD DISEASE PROTOTYPE MEMORY
# ============================================================

DISEASE_PROTOTYPE_PATH = os.path.join(
    ROOT,
    "disease_prototypes.pt"
)

assert os.path.exists(
    DISEASE_PROTOTYPE_PATH
), (
    "Disease prototype memory not found:\n"
    f"{DISEASE_PROTOTYPE_PATH}"
)

disease_prototype_artifact = torch.load(
    DISEASE_PROTOTYPE_PATH,
    map_location="cpu",
    weights_only=False
)


# ============================================================
# 17. INSPECT PROTOTYPE ARTIFACT
# ============================================================

print("\nDisease prototype artifact type:")

print(
    type(
        disease_prototype_artifact
    )
)

if isinstance(
    disease_prototype_artifact,
    dict
):

    print(
        "Available keys:",
        list(
            disease_prototype_artifact.keys()
        )
    )

else:

    print(
        "Artifact is not a dictionary."
    )


# ============================================================
# 18. DRIVE ARTIFACT INVENTORY
# ============================================================

print("\n" + "=" * 80)
print("AVAILABLE DATASET ARTIFACTS")
print("=" * 80)

all_files = sorted(
    glob.glob(
        os.path.join(
            ROOT,
            "*"
        )
    )
)

for path in all_files:

    if os.path.isfile(path):

        size_mb = (
            os.path.getsize(path)
            /
            (1024 ** 2)
        )

        print(
            f"{os.path.basename(path):65s}"
            f"{size_mb:10.2f} MB"
        )


# ============================================================
# 19. SEARCH FOR RELEVANT METADATA FILES
# ============================================================

print("\n" + "=" * 80)
print("RELEVANT FUSION-RELATED FILES")
print("=" * 80)

relevant_keywords = [
    "stage1",
    "stage2",
    "quality",
    "feature",
    "prototype",
    "disease",
    "bilateral",
    "cross_eye",
    "train",
    "val",
    "validation"
]

for path in all_files:

    filename_lower = (
        os.path.basename(path).lower()
    )

    if any(
        keyword in filename_lower
        for keyword in relevant_keywords
    ):

        print(
            os.path.basename(path)
        )


# ============================================================
# 20. BASIC CROSS-EYE FINITE-VALUE CHECK
# ============================================================

for tensor in [
    train_cross_right,
    train_cross_left,
    train_cross_patient,
    val_cross_right,
    val_cross_left,
    val_cross_patient,
    train_cross_right_attention,
    train_cross_left_attention,
    val_cross_right_attention,
    val_cross_left_attention
]:

    assert torch.isfinite(
        tensor
    ).all()


# ============================================================
# 21. PATIENT-ID VERIFICATION
# ============================================================

assert len(
    train_patient_ids
) == EXPECTED_TRAIN_PATIENTS

assert len(
    val_patient_ids
) == EXPECTED_VAL_PATIENTS

assert len(
    set(train_patient_ids)
) == EXPECTED_TRAIN_PATIENTS

assert len(
    set(val_patient_ids)
) == EXPECTED_VAL_PATIENTS

assert set(
    train_patient_ids
).isdisjoint(
    set(val_patient_ids)
)


# ============================================================
# 22. FINAL STATUS
# ============================================================

print("\n" + "=" * 80)
print("CELL 1 INITIALIZATION PASSED")
print("=" * 80)

print("\nFrozen Cross-Eye representations:")
print(
    "  Train Right:",
    tuple(train_cross_right.shape)
)

print(
    "  Train Left:",
    tuple(train_cross_left.shape)
)

print(
    "  Train Patient:",
    tuple(train_cross_patient.shape)
)

print(
    "  Val Right:",
    tuple(val_cross_right.shape)
)

print(
    "  Val Left:",
    tuple(val_cross_left.shape)
)

print(
    "  Val Patient:",
    tuple(val_cross_patient.shape)
)

print(
    "\nQuality-score rows:",
    len(stage1_quality_df)
)

print(
    "Quality score range:",
    (
        stage1_quality_df[
            "quality_score"
        ].min(),
        stage1_quality_df[
            "quality_score"
        ].max()
    )
)

print(
    "\nTrain patients:",
    len(train_patient_ids)
)

print(
    "Validation patients:",
    len(val_patient_ids)
)

print(
    "\nCross-Eye checkpoint:",
    cross_eye_train.get(
        "model",
        "metadata not stored"
    )
)

print(
    "\nNo training performed."
)

print(
    "Cross-Eye remains frozen."
)

print(
    "Ready for Cell 2: artifact/alignment inspection."
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ADAPTIVE BILATERAL FUSION — ENVIRONMENT INITIALIZATION
Device: cuda
GPU: Tesla T4

Dataset root:
/content/drive/My Drive/Eye Disease/Dataset

Configuration:
Feature dimension: 768
Number of diseases: 8
Disease labels: ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
Expected training patients: 2141
Expected validation patients: 438

Stage-1 feature artifact type:
<class 'dict'>
Available keys: ['filenames', 'features', 'labels', 'quality_scores', 'label_columns', 'feature_dim']

Disease prototype artifact type:
<class 'dict'>
Available keys: ['disease_prototypes', 'label_columns', 'prototype_counts', 'feature_dim', 'num_prototypes', 'construction_method']

AVAILABLE DATASET ARTIFACTS
bilateral_no_cross_eye_baseline_best.pt                                2.45 MB
bilateral_no_cross_eye_validation_predictions.csv                      0.06 MB
cross_eye_bilateral_attentio

CELL 2 — INSPECT AND VERIFY ADAPTIVE-FUSION SOURCE ARTIFACTS

This cell inspects all previously generated artifacts required for
Adaptive Bilateral Fusion.

No training is performed.

The purpose is to establish the exact structure, dimensions, ordering,
and metadata available for:

    1. Cross-Eye representations
    2. Disease-Aware features
    3. Disease-Aware attention/relevance
    4. Image quality scores
    5. Bilateral patient metadata
    6. Stage-2 patient dataframes

The Adaptive Fusion module requires reliable eye-level alignment.

For every bilateral patient we ultimately need:

    Right:
        Cross-Eye representation
        Quality score
        Disease relevance

    Left:
        Cross-Eye representation
        Quality score
        Disease relevance

The patient ID and eye filenames will be used to verify alignment.

This cell does NOT construct the final fusion dataset yet.
It only inspects and verifies the available source artifacts.

CELL 2B — VERIFY TRAIN/VALIDATION DAA ALIGNMENT

The previous inspection established that:

    Training DAA features:
        [4491, 768]

    Training DAA attention:
        [4491, 8]

    Validation DAA features:
        Right [438, 768]
        Left  [438, 768]

    Validation DAA attention:
        Right [438, 8]
        Left  [438, 8]

Training DAA artifacts are image-level, whereas the bilateral
metadata is patient-level.

Therefore, before constructing Adaptive Fusion memory, we must
deterministically map each training image representation to its
corresponding bilateral patient and eye.

The mapping will be verified using the actual Stage-1 filename
metadata.

Validation already contains explicit right/left DAA tensors and
therefore requires no reconstruction.

No model training is performed in this cell.

In [4]:
import os
import pandas as pd
import torch

# ============================================================
# CELL 2 — CORRECTED DAA ALIGNMENT
# ============================================================

print("=" * 80)
print("ADAPTIVE BILATERAL FUSION — DAA ALIGNMENT VERIFICATION")
print("=" * 80)


# ============================================================
# 1. LOAD STAGE-1 TRAINING DATAFRAME
# ============================================================

STAGE1_TRAIN_DF_PATH = os.path.join(
    ROOT,
    "stage1_train_df.csv"
)

assert os.path.exists(
    STAGE1_TRAIN_DF_PATH
)

stage1_train_source_df = pd.read_csv(
    STAGE1_TRAIN_DF_PATH
)

print(
    "\nStage-1 training dataframe:",
    stage1_train_source_df.shape
)

print(
    "Columns:",
    stage1_train_source_df.columns.tolist()
)


# ============================================================
# ADDED: LOAD STAGE-2 BILATERAL METADATA
# ============================================================

TRAIN_BILATERAL_METADATA_PATH = os.path.join(
    ROOT,
    "stage2_train_bilateral_metadata.csv"
)

VAL_BILATERAL_METADATA_PATH = os.path.join(
    ROOT,
    "stage2_val_bilateral_metadata.csv"
)

assert os.path.exists(TRAIN_BILATERAL_METADATA_PATH), \
    f"Training bilateral metadata not found: {TRAIN_BILATERAL_METADATA_PATH}"
assert os.path.exists(VAL_BILATERAL_METADATA_PATH), \
    f"Validation bilateral metadata not found: {VAL_BILATERAL_METADATA_PATH}"

train_bilateral_metadata = pd.read_csv(TRAIN_BILATERAL_METADATA_PATH)
val_bilateral_metadata = pd.read_csv(VAL_BILATERAL_METADATA_PATH)

print(
    "\nTraining bilateral metadata:",
    train_bilateral_metadata.shape
)

print(
    "Validation bilateral metadata:",
    val_bilateral_metadata.shape
)


# ============================================================
# ADDED: ASSIGN TRAIN DAA ARTIFACTS
# ============================================================
# `stage1_train_features_artifact` is loaded in Cell 1, Section 14.

train_daa_artifact = stage1_train_features_artifact['features']

TRAIN_DAA_ATTENTION_PATH = os.path.join(
    ROOT,
    "disease_attention_weights.pt"
)

train_daa_attention_artifact = torch.load(
    TRAIN_DAA_ATTENTION_PATH,
    map_location="cpu",
    weights_only=False
)


# ============================================================
# ADDED: LOAD VAL DAA ARTIFACTS
# ============================================================
VAL_DAA_FEATURES_PATH = os.path.join(
    ROOT,
    "stage2_val_disease_aware_features.pt"
)
VAL_DAA_ATTENTION_PATH = os.path.join(
    ROOT,
    "stage2_val_disease_attention_weights.pt"
)

assert os.path.exists(VAL_DAA_FEATURES_PATH), \
    f"Validation DAA features not found: {VAL_DAA_FEATURES_PATH}"
assert os.path.exists(VAL_DAA_ATTENTION_PATH), \
    f"Validation DAA attention not found: {VAL_DAA_ATTENTION_PATH}"

val_daa_artifact = torch.load(VAL_DAA_FEATURES_PATH, map_location="cpu", weights_only=False)
val_daa_attention_artifact = torch.load(VAL_DAA_ATTENTION_PATH, map_location="cpu", weights_only=False)

print(
    "\nValidation DAA features artifact type:",
    type(val_daa_artifact)
)
if isinstance(val_daa_artifact, dict):
    print(
        "Validation DAA features keys:",
        list(val_daa_artifact.keys())
    )

print(
    "\nValidation DAA attention artifact type:",
    type(val_daa_attention_artifact)
)
if isinstance(val_daa_attention_artifact, dict):
    print(
        "Validation DAA attention keys:",
        list(val_daa_attention_artifact.keys())
    )

# ============================================================
# 2. VERIFY STAGE-1 DAA ARTIFACTS
# ============================================================

assert torch.is_tensor(
    train_daa_artifact
)

assert torch.is_tensor(
    train_daa_attention_artifact
)

assert train_daa_artifact.shape == (
    4491,
    FEATURE_DIM
)

assert train_daa_attention_artifact.shape == (
    4491,
    NUM_CLASSES
)

print(
    "\nStage-1 DAA features:",
    tuple(train_daa_artifact.shape)
)

print(
    "Stage-1 DAA attention:",
    tuple(train_daa_attention_artifact.shape)
)


# ============================================================
# 3. IDENTIFY FILENAME COLUMN
# ============================================================

assert "filename" in (
    stage1_train_source_df.columns
)

STAGE1_FILENAME_COL = "filename"

stage1_train_filenames = (
    stage1_train_source_df[
        STAGE1_FILENAME_COL
    ]
    .astype(str)
    .tolist()
)

assert len(
    stage1_train_filenames
) == 4491

assert len(
    set(stage1_train_filenames)
) == 4491

print(
    "\nStage-1 image rows:",
    len(stage1_train_filenames)
)

print(
    "Unique Stage-1 filenames:",
    len(set(stage1_train_filenames))
)


# ============================================================
# 4. GET BILATERAL TRAINING FILENAMES
# ============================================================

assert (
    "right_filename"
    in train_bilateral_metadata.columns
)

assert (
    "left_filename"
    in train_bilateral_metadata.columns
)

train_right_filenames = (
    train_bilateral_metadata[
        "right_filename"
    ]
    .astype(str)
    .tolist()
)

train_left_filenames = (
    train_bilateral_metadata[
        "left_filename"
    ]
    .astype(str)
    .tolist()
)

assert len(
    train_right_filenames
) == EXPECTED_TRAIN_PATIENTS

assert len(
    train_left_filenames
) == EXPECTED_TRAIN_PATIENTS

print(
    "\nBilateral right-eye images:",
    len(train_right_filenames)
)

print(
    "Bilateral left-eye images:",
    len(train_left_filenames)
)

print(
    "Total bilateral eye images:",
    len(train_right_filenames)
    +
    len(train_left_filenames)
)


# ============================================================
# 5. EXPECTED BILATERAL IMAGE COUNT
# ============================================================

EXPECTED_BILATERAL_TRAIN_IMAGES = (
    EXPECTED_TRAIN_PATIENTS * 2
)

assert (
    EXPECTED_BILATERAL_TRAIN_IMAGES
    ==
    4282
)

assert (
    len(train_right_filenames)
    +
    len(train_left_filenames)
    ==
    EXPECTED_BILATERAL_TRAIN_IMAGES
)


# ============================================================
# 6. CREATE FILENAME SETS
# ============================================================

stage1_filename_set = set(
    stage1_train_filenames
)

bilateral_right_filename_set = set(
    train_right_filenames
)

bilateral_left_filename_set = set(
    train_left_filenames
)

bilateral_train_filename_set = (
    bilateral_right_filename_set
    |
    bilateral_left_filename_set
)


# ============================================================
# 7. VERIFY RIGHT/LEFT FILENAMES ARE UNIQUE
# ============================================================

assert len(
    bilateral_right_filename_set
) == EXPECTED_TRAIN_PATIENTS

assert len(
    bilateral_left_filename_set
) == EXPECTED_TRAIN_PATIENTS


# No image can simultaneously be the right and left image.
assert (
    bilateral_right_filename_set
    .isdisjoint(
        bilateral_left_filename_set
    )
)


# ============================================================
# 8. VERIFY BILATERAL IMAGE COUNT
# ============================================================

assert len(
    bilateral_train_filename_set
) == EXPECTED_BILATERAL_TRAIN_IMAGES

print(
    "\nUnique bilateral training filenames:",
    len(bilateral_train_filename_set)
)


# ============================================================
# 9. CHECK BILATERAL → STAGE-1 COVERAGE
# ============================================================

missing_from_stage1 = (
    bilateral_train_filename_set
    -
    stage1_filename_set
)

assert len(
    missing_from_stage1
) == 0, (
    "Bilateral image(s) are missing from "
    "Stage-1 DAA artifacts."
)

print(
    "\n✓ Every bilateral eye image exists "
    "in Stage-1 DAA artifacts."
)


# ============================================================
# 10. IDENTIFY STAGE-1 IMAGES NOT USED IN BILATERAL FUSION
# ============================================================

stage1_non_bilateral = (
    stage1_filename_set
    -
    bilateral_train_filename_set
)

print(
    "\nStage-1 images:",
    len(stage1_filename_set)
)

print(
    "Bilateral Stage-2 images:",
    len(bilateral_train_filename_set)
)

print(
    "Stage-1 images outside bilateral cohort:",
    len(stage1_non_bilateral)
)

assert (
    len(stage1_non_bilateral)
    ==
    4491 - 4282
)

assert (
    len(stage1_non_bilateral)
    ==
    209
)


# ============================================================
# 11. IMPORTANT DATASET INTEGRITY RESULT
# ============================================================

print(
    "\nDataset cohort relationship:"
)

print(
    "  Stage-1 image-level cohort : 4491"
)

print(
    "  Stage-2 bilateral images   : 4282"
)

print(
    "  Excluded non-bilateral     : 209"
)

print(
    "\nThe 209 non-bilateral Stage-1 images "
    "will NOT enter Adaptive Fusion."
)


# ============================================================
# 12. CREATE STAGE-1 FILENAME → DAA INDEX MAP
# ============================================================

train_filename_to_daa_index = {
    filename: index
    for index, filename
    in enumerate(
        stage1_train_filenames
    )
}

assert len(
    train_filename_to_daa_index
) == 4491


# ============================================================
# 13. MAP RIGHT EYES
# ============================================================

train_right_daa_indices = [
    train_filename_to_daa_index[
        filename
    ]
    for filename in train_right_filenames
]


# ============================================================
# 14. MAP LEFT EYES
# ============================================================

train_left_daa_indices = [
    train_filename_to_daa_index[
        filename
    ]
    for filename in train_left_filenames
]


# ============================================================
# 15. VERIFY INDEX COUNTS
# ============================================================

assert len(
    train_right_daa_indices
) == EXPECTED_TRAIN_PATIENTS

assert len(
    train_left_daa_indices
) == EXPECTED_TRAIN_PATIENTS


# ============================================================
# 16. VERIFY ALL BILATERAL INDICES ARE UNIQUE
# ============================================================

all_bilateral_daa_indices = (
    train_right_daa_indices
    +
    train_left_daa_indices
)

assert len(
    all_bilateral_daa_indices
) == 4282

assert len(
    set(all_bilateral_daa_indices)
) == 4282


# ============================================================
# 17. VERIFY MAPPED INDICES ARE VALID
# ============================================================

assert all(
    0 <= index < 4491
    for index in all_bilateral_daa_indices
)


# ============================================================
# 18. EXTRACT TRAINING RIGHT/LEFT DAA FEATURES
# ============================================================

train_daa_features_tensor = (
    train_daa_artifact.float()
)

train_daa_attention_tensor = (
    train_daa_attention_artifact.float()
)


train_right_daa_features = (
    train_daa_features_tensor[
        train_right_daa_indices
    ]
)

train_left_daa_features = (
    train_daa_features_tensor[
        train_left_daa_indices
    ]
)


# ============================================================
# 19. EXTRACT DISEASE RELEVANCE
# ============================================================

train_right_disease_attention = (
    train_daa_attention_tensor[
        train_right_daa_indices
    ]
)

train_left_disease_attention = (
    train_daa_attention_tensor[
        train_left_daa_indices
    ]
)


# ============================================================
# 20. VERIFY TRAINING DAA SHAPES
# ============================================================

assert train_right_daa_features.shape == (
    EXPECTED_TRAIN_PATIENTS,
    FEATURE_DIM
)

assert train_left_daa_features.shape == (
    EXPECTED_TRAIN_PATIENTS,
    FEATURE_DIM
)

assert train_right_disease_attention.shape == (
    EXPECTED_TRAIN_PATIENTS,
    NUM_CLASSES
)

assert train_left_disease_attention.shape == (
    EXPECTED_TRAIN_PATIENTS,
    NUM_CLASSES
)


# ============================================================
# 21. LOAD VALIDATION DAA ARTIFACTS
# ============================================================

assert isinstance(
    val_daa_artifact,
    dict
)

assert isinstance(
    val_daa_attention_artifact,
    dict
)

val_right_daa_features = (
    val_daa_artifact[
        "right_features"
    ].float()
)

val_left_daa_features = (
    val_daa_artifact[
        "left_features"
    ].float()
)

val_right_disease_attention = (
    val_daa_attention_artifact[
        "right_attention"
    ].float()
)

val_left_disease_attention = (
    val_daa_attention_artifact[
        "left_attention"
    ].float()
)


# ============================================================
# 22. VERIFY VALIDATION DAA SHAPES
# ============================================================

assert val_right_daa_features.shape == (
    EXPECTED_VAL_PATIENTS,
    FEATURE_DIM
)

assert val_left_daa_features.shape == (
    EXPECTED_VAL_PATIENTS,
    FEATURE_DIM
)

assert val_right_disease_attention.shape == (
    EXPECTED_VAL_PATIENTS,
    NUM_CLASSES
)

assert val_left_disease_attention.shape == (
    EXPECTED_VAL_PATIENTS,
    NUM_CLASSES
)


# ============================================================
# 23. LOAD QUALITY FROM BILATERAL METADATA
# ============================================================

train_right_quality = torch.tensor(
    train_bilateral_metadata[
        "right_quality"
    ].to_numpy(
        dtype=np.float32
    )
)

train_left_quality = torch.tensor(
    train_bilateral_metadata[
        "left_quality"
    ].to_numpy(
        dtype=np.float32
    )
)

val_right_quality = torch.tensor(
    val_bilateral_metadata[
        "right_quality"
    ].to_numpy(
        dtype=np.float32
    )
)

val_left_quality = torch.tensor(
    val_bilateral_metadata[
        "left_quality"
    ].to_numpy(
        dtype=np.float32
    )
)


# ============================================================
# 24. QUALITY SHAPE CHECKS
# ============================================================

assert train_right_quality.shape == (
    EXPECTED_TRAIN_PATIENTS,
)

assert train_left_quality.shape == (
    EXPECTED_TRAIN_PATIENTS,
)

assert val_right_quality.shape == (
    EXPECTED_VAL_PATIENTS,
)

assert val_left_quality.shape == (
    EXPECTED_VAL_PATIENTS,
)


# ============================================================
# 25. QUALITY RANGE CHECKS
# ============================================================

for quality_tensor in [
    train_right_quality,
    train_left_quality,
    val_right_quality,
    val_left_quality
]:

    assert torch.isfinite(
        quality_tensor
    ).all()

    assert bool(
        (
            (quality_tensor >= 0)
            &
            (quality_tensor <= 1)
        ).all()
    )


# ============================================================
# 26. FINITE-VALUE CHECKS
# ============================================================

for tensor in [
    train_right_daa_features,
    train_left_daa_features,
    train_right_disease_attention,
    train_left_disease_attention,
    val_right_daa_features,
    val_left_daa_features,
    val_right_disease_attention,
    val_left_disease_attention
]:

    assert torch.isfinite(
        tensor
    ).all()


# ============================================================
# 27. VERIFY VALIDATION PATIENT ORDER
# ============================================================

assert (
    val_bilateral_metadata[
        "patient_id"
    ].tolist()
    ==
    val_patient_ids
)


# ============================================================
# 28. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("DAA ALIGNMENT VERIFICATION PASSED")
print("=" * 80)

print("\nTRAINING")

print(
    "  Stage-1 images:",
    len(stage1_train_filenames)
)

print(
    "  Bilateral patients:",
    EXPECTED_TRAIN_PATIENTS
)

print(
    "  Bilateral eye images:",
    EXPECTED_BILATERAL_TRAIN_IMAGES
)

print(
    "  Excluded non-bilateral images:",
    len(stage1_non_bilateral)
)

print(
    "  Right DAA features:",
    tuple(train_right_daa_features.shape)
)

print(
    "  Left DAA features:",
    tuple(train_left_daa_features.shape)
)

print(
    "  Right disease relevance:",
    tuple(train_right_disease_attention.shape)
)

print(
    "  Left disease relevance:",
    tuple(train_left_disease_attention.shape)
)

print(
    "  Right quality:",
    tuple(train_right_quality.shape)
)

print(
    "  Left quality:",
    tuple(train_left_quality.shape)
)


print("\nVALIDATION")

print(
    "  Bilateral patients:",
    EXPECTED_VAL_PATIENTS
)

print(
    "  Right DAA features:",
    tuple(val_right_daa_features.shape)
)

print(
    "  Left DAA features:",
    tuple(val_left_daa_features.shape)
)

print(
    "  Right disease relevance:",
    tuple(val_right_disease_attention.shape)
)

print(
    "  Left disease relevance:",
    tuple(val_left_disease_attention.shape)
)

print(
    "  Right quality:",
    tuple(val_right_quality.shape)
)

print(
    "  Left quality:",
    tuple(val_left_quality.shape)
)


print("\nMapping integrity:")

print(
    "  ✓ 4282 bilateral eye images identified"
)

print(
    "  ✓ 4282 bilateral images found in Stage-1"
)

print(
    "  ✓ All bilateral images mapped exactly once"
)

print(
    "  ✓ 209 non-bilateral Stage-1 images correctly excluded"
)

print(
    "  ✓ Right/left eye identity preserved"
)

print(
    "  ✓ Bilateral patient ordering preserved"
)

print(
    "  ✓ Validation ordering preserved"
)

print(
    "  ✓ Disease relevance aligned"
)

print(
    "  ✓ Quality aligned"
)

print(
    "  ✓ All tensors finite"
)

print(
    "\nAdaptive Fusion source alignment is READY."
)

ADAPTIVE BILATERAL FUSION — DAA ALIGNMENT VERIFICATION

Stage-1 training dataframe: (4491, 13)
Columns: ['patient_id', 'eye', 'filename', 'image_path', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O', 'sample_weight']

Training bilateral metadata: (2141, 13)
Validation bilateral metadata: (438, 13)

Validation DAA features artifact type: <class 'dict'>
Validation DAA features keys: ['right_features', 'left_features']

Validation DAA attention artifact type: <class 'dict'>
Validation DAA attention keys: ['right_attention', 'left_attention']

Stage-1 DAA features: (4491, 768)
Stage-1 DAA attention: (4491, 8)

Stage-1 image rows: 4491
Unique Stage-1 filenames: 4491

Bilateral right-eye images: 2141
Bilateral left-eye images: 2141
Total bilateral eye images: 4282

Unique bilateral training filenames: 4282

✓ Every bilateral eye image exists in Stage-1 DAA artifacts.

Stage-1 images: 4491
Bilateral Stage-2 images: 4282
Stage-1 images outside bilateral cohort: 209

Dataset cohort relationship:
  Stag

In [5]:
print("=" * 80)
print("TRAIN DAA ATTENTION SOURCE CHECK")
print("=" * 80)

print("\ntrain_daa_attention_artifact:")
print(type(train_daa_attention_artifact))
print("Shape:", train_daa_attention_artifact.shape)
print("First 3:")
print(train_daa_attention_artifact[:3])

print("\nStage-1 dataframe labels:")
print(
    torch.tensor(
        stage1_train_source_df[LABEL_COLUMNS].values[:3],
        dtype=torch.float32
    )
)

print("\nExtracted RIGHT DAA attention:")
print(train_right_disease_attention[:3])

print("\nExtracted LEFT DAA attention:")
print(train_left_disease_attention[:3])

print("\nRight vs Left absolute difference:")
print(
    torch.abs(
        train_right_disease_attention -
        train_left_disease_attention
    )[:3]
)

print(
    "\nMaximum R-L attention difference:",
    torch.abs(
        train_right_disease_attention -
        train_left_disease_attention
    ).max().item()
)

print(
    "\nRight attention min/max:",
    train_right_disease_attention.min().item(),
    train_right_disease_attention.max().item()
)

print(
    "Left attention min/max:",
    train_left_disease_attention.min().item(),
    train_left_disease_attention.max().item()
)

print("=" * 80)

TRAIN DAA ATTENTION SOURCE CHECK

train_daa_attention_artifact:
<class 'torch.Tensor'>
Shape: torch.Size([4491, 8])
First 3:
tensor([[0.2483, 0.2151, 0.0708, 0.0581, 0.0560, 0.0653, 0.0603, 0.2260],
        [0.2456, 0.2111, 0.0660, 0.0584, 0.0587, 0.0592, 0.0648, 0.2361],
        [0.1693, 0.2693, 0.0472, 0.0765, 0.0604, 0.0957, 0.0655, 0.2161]])

Stage-1 dataframe labels:
tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 1.]])

Extracted RIGHT DAA attention:
tensor([[0.2483, 0.2151, 0.0708, 0.0581, 0.0560, 0.0653, 0.0603, 0.2260],
        [0.1960, 0.2565, 0.0531, 0.0748, 0.0531, 0.0814, 0.0638, 0.2212],
        [0.1605, 0.2630, 0.0499, 0.0798, 0.0752, 0.1051, 0.0572, 0.2094]])

Extracted LEFT DAA attention:
tensor([[0.2456, 0.2111, 0.0660, 0.0584, 0.0587, 0.0592, 0.0648, 0.2361],
        [0.2052, 0.2480, 0.0504, 0.0660, 0.0596, 0.0704, 0.0618, 0.2386],
        [0.2611, 0.1848, 0.1030, 0.0689, 0.0609, 0.0690, 0.0540,

CELL 3 — BUILD ADAPTIVE BILATERAL FUSION MEMORY

The source alignment has been verified.

This cell now constructs the final frozen patient-level memory that
will be consumed by the Adaptive Bilateral Fusion module.

For every bilateral patient we retain:

    RIGHT EYE
        • Cross-Eye representation [768]
        • Quality score [1]
        • Disease relevance [8]

    LEFT EYE
        • Cross-Eye representation [768]
        • Quality score [1]
        • Disease relevance [8]

    PATIENT REFERENCE
        • Frozen Cross-Eye patient representation [768]

The two eye representations remain separate.

The Adaptive Fusion module will later learn scalar contribution
weights for the right and left eyes.

The disease relevance tensor contains one relevance value per
disease and is therefore retained as an 8-dimensional signal.

No fusion weights are learned in this cell.

No representations are modified.

No validation information is used to construct training statistics.

The resulting memory is saved as a deterministic artifact so that
the Adaptive Fusion experiments can be reproduced exactly.

In [6]:
# ============================================================
# CELL 3 — BUILD ADAPTIVE BILATERAL FUSION MEMORY
# ============================================================

print("=" * 80)
print("ADAPTIVE BILATERAL FUSION — BUILD FROZEN MEMORY")
print("=" * 80)


# ============================================================
# 1. VERIFY ALL REQUIRED VARIABLES EXIST
# ============================================================

required_variables = [
    "train_cross_right",
    "train_cross_left",
    "train_cross_patient",
    "val_cross_right",
    "val_cross_left",
    "val_cross_patient",

    "train_right_daa_features",
    "train_left_daa_features",
    "val_right_daa_features",
    "val_left_daa_features",

    "train_right_disease_attention",
    "train_left_disease_attention",
    "val_right_disease_attention",
    "val_left_disease_attention",

    "train_right_quality",
    "train_left_quality",
    "val_right_quality",
    "val_left_quality",

    "train_patient_ids",
    "val_patient_ids"
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

assert not missing_variables, (
    "Missing required variables: "
    + str(missing_variables)
)

print(
    "\n✓ All required source tensors are available."
)


# ============================================================
# 2. CREATE TRAINING FUSION MEMORY
# ============================================================

fusion_train_memory = {
    "right_cross_eye":
        train_cross_right.clone(),

    "left_cross_eye":
        train_cross_left.clone(),

    "patient_cross_eye":
        train_cross_patient.clone(),

    "right_quality":
        train_right_quality.clone(),

    "left_quality":
        train_left_quality.clone(),

    "right_disease_relevance":
        train_right_disease_attention.clone(),

    "left_disease_relevance":
        train_left_disease_attention.clone(),

    "right_daa_features":
        train_right_daa_features.clone(),

    "left_daa_features":
        train_left_daa_features.clone(),

    "patient_ids":
        list(train_patient_ids),

    "num_patients":
        EXPECTED_TRAIN_PATIENTS,

    "feature_dim":
        FEATURE_DIM,

    "num_classes":
        NUM_CLASSES,

    "source":
        "Frozen Stage-2 Cross-Eye + Stage-1 DAA + bilateral quality"
}


# ============================================================
# 3. CREATE VALIDATION FUSION MEMORY
# ============================================================

fusion_val_memory = {
    "right_cross_eye":
        val_cross_right.clone(),

    "left_cross_eye":
        val_cross_left.clone(),

    "patient_cross_eye":
        val_cross_patient.clone(),

    "right_quality":
        val_right_quality.clone(),

    "left_quality":
        val_left_quality.clone(),

    "right_disease_relevance":
        val_right_disease_attention.clone(),

    "left_disease_relevance":
        val_left_disease_attention.clone(),

    "right_daa_features":
        val_right_daa_features.clone(),

    "left_daa_features":
        val_left_daa_features.clone(),

    "patient_ids":
        list(val_patient_ids),

    "num_patients":
        EXPECTED_VAL_PATIENTS,

    "feature_dim":
        FEATURE_DIM,

    "num_classes":
        NUM_CLASSES,

    "source":
        "Frozen Stage-2 Cross-Eye + Stage-1 DAA + bilateral quality"
}


# ============================================================
# 4. VERIFY MEMORY KEYS
# ============================================================

expected_memory_keys = [
    "right_cross_eye",
    "left_cross_eye",
    "patient_cross_eye",
    "right_quality",
    "left_quality",
    "right_disease_relevance",
    "left_disease_relevance",
    "right_daa_features",
    "left_daa_features",
    "patient_ids"
]

for key in expected_memory_keys:

    assert key in fusion_train_memory
    assert key in fusion_val_memory


# ============================================================
# 5. VERIFY TRAINING MEMORY SHAPES
# ============================================================

assert fusion_train_memory[
    "right_cross_eye"
].shape == (
    EXPECTED_TRAIN_PATIENTS,
    FEATURE_DIM
)

assert fusion_train_memory[
    "left_cross_eye"
].shape == (
    EXPECTED_TRAIN_PATIENTS,
    FEATURE_DIM
)

assert fusion_train_memory[
    "patient_cross_eye"
].shape == (
    EXPECTED_TRAIN_PATIENTS,
    FEATURE_DIM
)

assert fusion_train_memory[
    "right_quality"
].shape == (
    EXPECTED_TRAIN_PATIENTS,
)

assert fusion_train_memory[
    "left_quality"
].shape == (
    EXPECTED_TRAIN_PATIENTS,
)

assert fusion_train_memory[
    "right_disease_relevance"
].shape == (
    EXPECTED_TRAIN_PATIENTS,
    NUM_CLASSES
)

assert fusion_train_memory[
    "left_disease_relevance"
].shape == (
    EXPECTED_TRAIN_PATIENTS,
    NUM_CLASSES
)


# ============================================================
# 6. VERIFY VALIDATION MEMORY SHAPES
# ============================================================

assert fusion_val_memory[
    "right_cross_eye"
].shape == (
    EXPECTED_VAL_PATIENTS,
    FEATURE_DIM
)

assert fusion_val_memory[
    "left_cross_eye"
].shape == (
    EXPECTED_VAL_PATIENTS,
    FEATURE_DIM
)

assert fusion_val_memory[
    "patient_cross_eye"
].shape == (
    EXPECTED_VAL_PATIENTS,
    FEATURE_DIM
)

assert fusion_val_memory[
    "right_quality"
].shape == (
    EXPECTED_VAL_PATIENTS,
)

assert fusion_val_memory[
    "left_quality"
].shape == (
    EXPECTED_VAL_PATIENTS,
)

assert fusion_val_memory[
    "right_disease_relevance"
].shape == (
    EXPECTED_VAL_PATIENTS,
    NUM_CLASSES
)

assert fusion_val_memory[
    "left_disease_relevance"
].shape == (
    EXPECTED_VAL_PATIENTS,
    NUM_CLASSES
)


# ============================================================
# 7. VERIFY ALL NUMERIC TENSORS ARE FINITE
# ============================================================

numeric_memory_keys = [
    "right_cross_eye",
    "left_cross_eye",
    "patient_cross_eye",
    "right_quality",
    "left_quality",
    "right_disease_relevance",
    "left_disease_relevance",
    "right_daa_features",
    "left_daa_features"
]

for memory in [
    fusion_train_memory,
    fusion_val_memory
]:

    for key in numeric_memory_keys:

        assert torch.isfinite(
            memory[key]
        ).all(), (
            f"Non-finite values found in {key}"
        )


# ============================================================
# 8. VERIFY QUALITY RANGE
# ============================================================

for memory in [
    fusion_train_memory,
    fusion_val_memory
]:

    for key in [
        "right_quality",
        "left_quality"
    ]:

        quality = memory[key]

        assert bool(
            (
                (quality >= 0)
                &
                (quality <= 1)
            ).all()
        )


# ============================================================
# 9. VERIFY PATIENT IDS
# ============================================================

assert len(
    fusion_train_memory[
        "patient_ids"
    ]
) == EXPECTED_TRAIN_PATIENTS

assert len(
    fusion_val_memory[
        "patient_ids"
    ]
) == EXPECTED_VAL_PATIENTS

assert len(
    set(
        fusion_train_memory[
            "patient_ids"
        ]
    )
) == EXPECTED_TRAIN_PATIENTS

assert len(
    set(
        fusion_val_memory[
            "patient_ids"
        ]
    )
) == EXPECTED_VAL_PATIENTS

assert set(
    fusion_train_memory[
        "patient_ids"
    ]
).isdisjoint(
    set(
        fusion_val_memory[
            "patient_ids"
        ]
    )
)


# ============================================================
# 10. VERIFY PATIENT ORDER
# ============================================================

assert (
    fusion_train_memory[
        "patient_ids"
    ]
    ==
    train_patient_ids
)

assert (
    fusion_val_memory[
        "patient_ids"
    ]
    ==
    val_patient_ids
)


# ============================================================
# 11. VERIFY QUALITY / DISEASE RELEVANCE ALIGNMENT
# ============================================================

# Quality comes directly from the Stage-2 bilateral metadata.

assert torch.equal(
    fusion_train_memory[
        "right_quality"
    ],
    train_right_quality
)

assert torch.equal(
    fusion_train_memory[
        "left_quality"
    ],
    train_left_quality
)

assert torch.equal(
    fusion_val_memory[
        "right_quality"
    ],
    val_right_quality
)

assert torch.equal(
    fusion_val_memory[
        "left_quality"
    ],
    val_left_quality
)


# ============================================================
# 12. COMPUTE SIMPLE QUALITY DIFFERENCE STATISTICS
# ============================================================

train_quality_difference = (
    fusion_train_memory[
        "right_quality"
    ]
    -
    fusion_train_memory[
        "left_quality"
    ]
)

val_quality_difference = (
    fusion_val_memory[
        "right_quality"
    ]
    -
    fusion_val_memory[
        "left_quality"
    ]
)


print("\n" + "=" * 80)
print("QUALITY DIFFERENCE STATISTICS")
print("=" * 80)

print("\nTraining:")
print(
    "  Mean R-L quality:",
    train_quality_difference.mean().item()
)

print(
    "  Std R-L quality:",
    train_quality_difference.std().item()
)

print(
    "  Mean |R-L| quality:",
    train_quality_difference.abs().mean().item()
)

print(
    "  Right higher-quality:",
    (
        train_quality_difference > 0
    ).float().mean().item()
)

print(
    "  Left higher-quality:",
    (
        train_quality_difference < 0
    ).float().mean().item()
)

print(
    "  Equal quality:",
    (
        train_quality_difference == 0
    ).float().mean().item()
)


print("\nValidation:")

print(
    "  Mean R-L quality:",
    val_quality_difference.mean().item()
)

print(
    "  Std R-L quality:",
    val_quality_difference.std().item()
)

print(
    "  Mean |R-L| quality:",
    val_quality_difference.abs().mean().item()
)

print(
    "  Right higher-quality:",
    (
        val_quality_difference > 0
    ).float().mean().item()
)

print(
    "  Left higher-quality:",
    (
        val_quality_difference < 0
    ).float().mean().item()
)

print(
    "  Equal quality:",
    (
        val_quality_difference == 0
    ).float().mean().item()
)


# ============================================================
# 13. DISEASE RELEVANCE SUMMARY
# ============================================================

train_relevance_difference = (
    fusion_train_memory[
        "right_disease_relevance"
    ]
    -
    fusion_train_memory[
        "left_disease_relevance"
    ]
)

val_relevance_difference = (
    fusion_val_memory[
        "right_disease_relevance"
    ]
    -
    fusion_val_memory[
        "left_disease_relevance"
    ]
)


print("\n" + "=" * 80)
print("DISEASE RELEVANCE SUMMARY")
print("=" * 80)

print("\nTraining:")

print(
    "  Mean absolute R-L attention difference:",
    train_relevance_difference.abs().mean().item()
)

print(
    "  Maximum absolute R-L attention difference:",
    train_relevance_difference.abs().max().item()
)

print(
    "  Mean right-eye attention:",
    fusion_train_memory[
        "right_disease_relevance"
    ].mean().item()
)

print(
    "  Mean left-eye attention:",
    fusion_train_memory[
        "left_disease_relevance"
    ].mean().item()
)


print("\nValidation:")

print(
    "  Mean absolute R-L attention difference:",
    val_relevance_difference.abs().mean().item()
)

print(
    "  Maximum absolute R-L attention difference:",
    val_relevance_difference.abs().max().item()
)

print(
    "  Mean right-eye attention:",
    fusion_val_memory[
        "right_disease_relevance"
    ].mean().item()
)

print(
    "  Mean left-eye attention:",
    fusion_val_memory[
        "left_disease_relevance"
    ].mean().item()
)
# ============================================================
# 14. SAVE FROZEN FUSION MEMORY
# ============================================================

FINAL_FUSION_TRAIN_MEMORY_PATH = os.path.join(
    ROOT,
    "stage2_train_adaptive_fusion_memory.pt"
)

FINAL_FUSION_VAL_MEMORY_PATH = os.path.join(
    ROOT,
    "stage2_val_adaptive_fusion_memory.pt"
)


torch.save(
    fusion_train_memory,
    FINAL_FUSION_TRAIN_MEMORY_PATH
)

torch.save(
    fusion_val_memory,
    FINAL_FUSION_VAL_MEMORY_PATH
)


# ============================================================
# 15. VERIFY FILE CREATION
# ============================================================

assert os.path.exists(
    FINAL_FUSION_TRAIN_MEMORY_PATH
)

assert os.path.exists(
    FINAL_FUSION_VAL_MEMORY_PATH
)


# ============================================================
# 16. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("ADAPTIVE BILATERAL FUSION MEMORY READY")
print("=" * 80)

print("\nTraining memory:")

print(
    "  Right Cross-Eye:",
    tuple(
        fusion_train_memory[
            "right_cross_eye"
        ].shape
    )
)

print(
    "  Left Cross-Eye:",
    tuple(
        fusion_train_memory[
            "left_cross_eye"
        ].shape
    )
)

print(
    "  Patient Cross-Eye:",
    tuple(
        fusion_train_memory[
            "patient_cross_eye"
        ].shape
    )
)

print(
    "  Right quality:",
    tuple(
        fusion_train_memory[
            "right_quality"
        ].shape
    )
)

print(
    "  Left quality:",
    tuple(
        fusion_train_memory[
            "left_quality"
        ].shape
    )
)

print(
    "  Right disease relevance:",
    tuple(
        fusion_train_memory[
            "right_disease_relevance"
        ].shape
    )
)

print(
    "  Left disease relevance:",
    tuple(
        fusion_train_memory[
            "left_disease_relevance"
        ].shape
    )
)


print("\nValidation memory:")

print(
    "  Right Cross-Eye:",
    tuple(
        fusion_val_memory[
            "right_cross_eye"
        ].shape
    )
)

print(
    "  Left Cross-Eye:",
    tuple(
        fusion_val_memory[
            "left_cross_eye"
        ].shape
    )
)

print(
    "  Patient Cross-Eye:",
    tuple(
        fusion_val_memory[
            "patient_cross_eye"
        ].shape
    )
)

print(
    "  Right quality:",
    tuple(
        fusion_val_memory[
            "right_quality"
        ].shape
    )
)

print(
    "  Left quality:",
    tuple(
        fusion_val_memory[
            "left_quality"
        ].shape
    )
)

print(
    "  Right disease relevance:",
    tuple(
        fusion_val_memory[
            "right_disease_relevance"
        ].shape
    )
)

print(
    "  Left disease relevance:",
    tuple(
        fusion_val_memory[
            "left_disease_relevance"
        ].shape
    )
)


print("\nSaved artifacts:")

print(
    "  Train:",
    FINAL_FUSION_TRAIN_MEMORY_PATH
)

print(
    "  Validation:",
    FINAL_FUSION_VAL_MEMORY_PATH
)


print("\nIntegrity:")
print("  ✓ Cross-Eye representations frozen")
print("  ✓ Patient representation preserved")
print("  ✓ Quality signals preserved")
print("  ✓ Disease relevance preserved")
print("  ✓ Right/left alignment preserved")
print("  ✓ Patient ordering preserved")
print("  ✓ Train/validation separated")
print("  ✓ All tensors finite")
print("  ✓ Fusion memory saved")
print("\nCell 3 PASSED.")

ADAPTIVE BILATERAL FUSION — BUILD FROZEN MEMORY

✓ All required source tensors are available.

QUALITY DIFFERENCE STATISTICS

Training:
  Mean R-L quality: 0.0021914388053119183
  Std R-L quality: 0.10499734431505203
  Mean |R-L| quality: 0.07022524625062943
  Right higher-quality: 0.5086408257484436
  Left higher-quality: 0.4913591742515564
  Equal quality: 0.0

Validation:
  Mean R-L quality: 0.004816655069589615
  Std R-L quality: 0.10642289370298386
  Mean |R-L| quality: 0.07311603426933289
  Right higher-quality: 0.5159817337989807
  Left higher-quality: 0.4840182662010193
  Equal quality: 0.0

DISEASE RELEVANCE SUMMARY

Training:
  Mean absolute R-L attention difference: 0.016374455764889717
  Maximum absolute R-L attention difference: 0.27109867334365845
  Mean right-eye attention: 0.125
  Mean left-eye attention: 0.125

Validation:
  Mean absolute R-L attention difference: 0.02543465606868267
  Maximum absolute R-L attention difference: 0.27725428342819214
  Mean right-eye atte

CELL 4 — ADAPTIVE BILATERAL FUSION MODULE

The frozen Cross-Eye representations are now used as the input
representation for Adaptive Bilateral Fusion.

The fusion module learns two patient-specific scalar logits:

    z_R
    z_L

These logits are converted into normalized eye contribution weights:

    [w_R, w_L] = Softmax([z_R, z_L])

Therefore:

    w_R + w_L = 1

The gate receives eye-level reliability information:

    • image quality
    • disease relevance

The gate does NOT modify the frozen Cross-Eye representation.

It only determines how strongly the right and left eye
representations contribute to the final patient representation.

The fused representation is:

    F_fused =
        w_R * Right_CrossEye
        +
        w_L * Left_CrossEye

The patient-level Cross-Eye representation is retained as an
additional reference representation.

IMPORTANT:

    • Cross-Eye parameters are frozen.
    • Stage-1 DAA parameters are frozen.
    • Only the Adaptive Fusion gate is trainable.
    • No validation data is used for parameter fitting.
    • Softmax guarantees normalized bilateral contribution weights.
    • The module is intentionally lightweight and interpretable.

Because the current saved disease-relevance tensors are empirically
almost uniform (approximately 1/8 per disease), this experiment will
also quantify whether disease relevance contributes meaningful
information beyond image quality.

In [7]:
# ============================================================
# CELL 4 — ADAPTIVE BILATERAL FUSION MODULE
# ============================================================

print("=" * 80)
print("ADAPTIVE BILATERAL FUSION — MODULE DEFINITION")
print("=" * 80)


# ============================================================
# 1. CONFIGURATION
# ============================================================

FUSION_FEATURE_DIM = FEATURE_DIM
FUSION_NUM_CLASSES = NUM_CLASSES

FUSION_HIDDEN_DIM = 32

print(
    "\nFusion feature dimension:",
    FUSION_FEATURE_DIM
)

print(
    "Disease relevance dimension:",
    FUSION_NUM_CLASSES
)

print(
    "Fusion hidden dimension:",
    FUSION_HIDDEN_DIM
)


# ============================================================
# 2. ADAPTIVE EYE GATE
# ============================================================

class AdaptiveEyeGate(nn.Module):

    def __init__(
        self,
        disease_dim=8,
        hidden_dim=32
    ):

        super().__init__()

        # Input:
        #   quality = 1
        #   disease relevance = 8
        #
        # Total = 9

        self.network = nn.Sequential(

            nn.Linear(
                1 + disease_dim,
                hidden_dim
            ),

            nn.LayerNorm(
                hidden_dim
            ),

            nn.GELU(),

            nn.Linear(
                hidden_dim,
                1
            )
        )


    def forward(
        self,
        quality,
        disease_relevance
    ):

        # ----------------------------------------------------
        # Ensure quality has shape [B, 1]
        # ----------------------------------------------------

        if quality.dim() == 1:

            quality = quality.unsqueeze(1)


        # ----------------------------------------------------
        # Concatenate reliability signals
        # ----------------------------------------------------

        gate_input = torch.cat(
            [
                quality,
                disease_relevance
            ],
            dim=1
        )


        # ----------------------------------------------------
        # Produce one scalar logit per eye
        # ----------------------------------------------------

        logit = self.network(
            gate_input
        )

        return logit


# ============================================================
# 3. ADAPTIVE BILATERAL FUSION
# ============================================================

class AdaptiveBilateralFusion(nn.Module):

    def __init__(
        self,
        feature_dim=768,
        disease_dim=8,
        hidden_dim=32
    ):

        super().__init__()

        self.feature_dim = feature_dim

        self.right_gate = AdaptiveEyeGate(
            disease_dim=disease_dim,
            hidden_dim=hidden_dim
        )

        self.left_gate = AdaptiveEyeGate(
            disease_dim=disease_dim,
            hidden_dim=hidden_dim
        )


    def forward(
        self,
        right_cross_eye,
        left_cross_eye,
        right_quality,
        left_quality,
        right_disease_relevance,
        left_disease_relevance
    ):

        # ====================================================
        # RIGHT EYE GATE
        # ====================================================

        right_logit = self.right_gate(
            right_quality,
            right_disease_relevance
        )


        # ====================================================
        # LEFT EYE GATE
        # ====================================================

        left_logit = self.left_gate(
            left_quality,
            left_disease_relevance
        )


        # ====================================================
        # BILATERAL SOFTMAX
        # ====================================================

        bilateral_logits = torch.cat(
            [
                right_logit,
                left_logit
            ],
            dim=1
        )

        bilateral_weights = F.softmax(
            bilateral_logits,
            dim=1
        )


        # ====================================================
        # EXTRACT EYE WEIGHTS
        # ====================================================

        right_weight = (
            bilateral_weights[:, 0:1]
        )

        left_weight = (
            bilateral_weights[:, 1:2]
        )


        # ====================================================
        # WEIGHTED BILATERAL FUSION
        # ====================================================

        fused_features = (
            right_weight
            *
            right_cross_eye
        ) + (
            left_weight
            *
            left_cross_eye
        )


        # ====================================================
        # RETURN ALL INTERPRETABLE COMPONENTS
        # ====================================================

        return {

            "fused_features":
                fused_features,

            "right_weight":
                right_weight,

            "left_weight":
                left_weight,

            "right_logit":
                right_logit,

            "left_logit":
                left_logit,

            "bilateral_weights":
                bilateral_weights
        }


# ============================================================
# 4. INSTANTIATE MODEL
# ============================================================

adaptive_fusion_model = (
    AdaptiveBilateralFusion(
        feature_dim=FUSION_FEATURE_DIM,
        disease_dim=FUSION_NUM_CLASSES,
        hidden_dim=FUSION_HIDDEN_DIM
    )
).to(device)


# ============================================================
# 5. MODEL SUMMARY
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "ADAPTIVE FUSION MODEL"
)

print(
    "=" * 80
)

print(
    adaptive_fusion_model
)


# ============================================================
# 6. PARAMETER SUMMARY
# ============================================================

total_parameters = sum(
    parameter.numel()
    for parameter
    in adaptive_fusion_model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter
    in adaptive_fusion_model.parameters()
    if parameter.requires_grad
)

print(
    "\nTotal parameters:",
    total_parameters
)

print(
    "Trainable parameters:",
    trainable_parameters
)


# ============================================================
# 7. VERIFY EXPECTED TRAINABLE STATUS
# ============================================================

assert (
    total_parameters
    ==
    trainable_parameters
)

assert (
    trainable_parameters
    > 0
)


# ============================================================
# 8. CREATE SMALL SYNTHETIC TEST
# ============================================================

TEST_BATCH = 8

test_right_features = torch.randn(
    TEST_BATCH,
    FEATURE_DIM,
    device=device
)

test_left_features = torch.randn(
    TEST_BATCH,
    FEATURE_DIM,
    device=device
)

test_right_quality = torch.rand(
    TEST_BATCH,
    device=device
)

test_left_quality = torch.rand(
    TEST_BATCH,
    device=device
)

test_right_relevance = F.softmax(
    torch.randn(
        TEST_BATCH,
        NUM_CLASSES,
        device=device
    ),
    dim=1
)

test_left_relevance = F.softmax(
    torch.randn(
        TEST_BATCH,
        NUM_CLASSES,
        device=device
    ),
    dim=1
)


# ============================================================
# 9. FORWARD PASS
# ============================================================

adaptive_fusion_model.eval()

with torch.no_grad():

    test_output = adaptive_fusion_model(
        test_right_features,
        test_left_features,
        test_right_quality,
        test_left_quality,
        test_right_relevance,
        test_left_relevance
    )


# ============================================================
# 10. VERIFY OUTPUT SHAPES
# ============================================================

assert test_output[
    "fused_features"
].shape == (
    TEST_BATCH,
    FEATURE_DIM
)

assert test_output[
    "right_weight"
].shape == (
    TEST_BATCH,
    1
)

assert test_output[
    "left_weight"
].shape == (
    TEST_BATCH,
    1
)

assert test_output[
    "right_logit"
].shape == (
    TEST_BATCH,
    1
)

assert test_output[
    "left_logit"
].shape == (
    TEST_BATCH,
    1
)

assert test_output[
    "bilateral_weights"
].shape == (
    TEST_BATCH,
    2
)


# ============================================================
# 11. VERIFY SOFTMAX NORMALIZATION
# ============================================================

weight_sum = (
    test_output[
        "right_weight"
    ]
    +
    test_output[
        "left_weight"
    ]
)

assert torch.allclose(
    weight_sum,
    torch.ones_like(weight_sum),
    atol=1e-6
)


# ============================================================
# 12. VERIFY WEIGHT RANGE
# ============================================================

assert bool(
    (
        test_output[
            "right_weight"
        ] >= 0
    ).all()
)

assert bool(
    (
        test_output[
            "right_weight"
        ] <= 1
    ).all()
)

assert bool(
    (
        test_output[
            "left_weight"
        ] >= 0
    ).all()
)

assert bool(
    (
        test_output[
            "left_weight"
        ] <= 1
    ).all()
)


# ============================================================
# 13. VERIFY FUSED REPRESENTATION
# ============================================================

expected_fused = (
    test_output[
        "right_weight"
    ]
    *
    test_right_features
) + (
    test_output[
        "left_weight"
    ]
    *
    test_left_features
)

assert torch.allclose(
    test_output[
        "fused_features"
    ],
    expected_fused,
    atol=1e-6
)


# ============================================================
# 14. VERIFY FINITE OUTPUTS
# ============================================================

for key, value in test_output.items():

    assert torch.isfinite(
        value
    ).all(), (
        f"Non-finite output: {key}"
    )


# ============================================================
# 15. RETURN TO TRAIN MODE
# ============================================================

adaptive_fusion_model.train()


# ============================================================
# 16. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "CELL 4 — ADAPTIVE FUSION MODULE PASSED"
)

print(
    "=" * 80
)

print(
    "\n✓ Adaptive right-eye gate defined"
)

print(
    "✓ Adaptive left-eye gate defined"
)

print(
    "✓ Quality signal accepted"
)

print(
    "✓ Disease relevance accepted"
)

print(
    "✓ Bilateral Softmax weighting verified"
)

print(
    "✓ Right weight + Left weight = 1"
)

print(
    "✓ Fused representation verified"
)

print(
    "✓ Output dimensions verified"
)

print(
    "✓ All outputs finite"
)

print(
    "\nNo training performed."
)

print(
    "Ready for Cell 5: baseline/adaptive-fusion experiment setup."
)

ADAPTIVE BILATERAL FUSION — MODULE DEFINITION

Fusion feature dimension: 768
Disease relevance dimension: 8
Fusion hidden dimension: 32

ADAPTIVE FUSION MODEL
AdaptiveBilateralFusion(
  (right_gate): AdaptiveEyeGate(
    (network): Sequential(
      (0): Linear(in_features=9, out_features=32, bias=True)
      (1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Linear(in_features=32, out_features=1, bias=True)
    )
  )
  (left_gate): AdaptiveEyeGate(
    (network): Sequential(
      (0): Linear(in_features=9, out_features=32, bias=True)
      (1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Linear(in_features=32, out_features=1, bias=True)
    )
  )
)

Total parameters: 834
Trainable parameters: 834

CELL 4 — ADAPTIVE FUSION MODULE PASSED

✓ Adaptive right-eye gate defined
✓ Adaptive left-eye gate defined
✓ Quality signal accepted
✓ Disease relevance accepted
✓ Bilateral Softmax

CELL 5 — PREPARE CONTROLLED ADAPTIVE FUSION EXPERIMENTS

This cell establishes the experimental protocol for evaluating
Adaptive Bilateral Fusion.

The Cross-Eye representations remain frozen.

Three fusion strategies will be compared:

    EXPERIMENT A — MEAN FUSION

        F_mean = (Right + Left) / 2

        No trainable fusion parameters.

    EXPERIMENT B — QUALITY-ONLY ADAPTIVE FUSION

        Eye contribution weights are learned from:

            Right quality
            Left quality

        Disease relevance is excluded.

    EXPERIMENT C — QUALITY + DISEASE ADAPTIVE FUSION

        Eye contribution weights are learned from:

            Right quality
            Left quality
            Right disease relevance
            Left disease relevance

The same downstream prediction head will be used for all learned
fusion experiments.

The purpose is to determine whether adaptive bilateral weighting
provides measurable benefit over simple averaging, and whether the
currently stored disease-relevance signal provides additional value
beyond image quality.

The patient-level Cross-Eye representation is retained as a separate
reference baseline.

No validation data is used for training.

No fusion model is trained in this cell.

In [8]:
# ============================================================
# CELL 5 — PREPARE CONTROLLED FUSION EXPERIMENTS
# ============================================================

print("=" * 80)
print("ADAPTIVE BILATERAL FUSION — EXPERIMENT PROTOCOL")
print("=" * 80)


# ============================================================
# 1. LOAD TARGET LABELS
# ============================================================

fusion_train_labels = torch.tensor(
    train_bilateral_metadata[
        LABEL_COLUMNS
    ].to_numpy(
        dtype=np.float32
    )
)

fusion_val_labels = torch.tensor(
    val_bilateral_metadata[
        LABEL_COLUMNS
    ].to_numpy(
        dtype=np.float32
    )
)


# ============================================================
# 2. VERIFY LABEL SHAPES
# ============================================================

assert fusion_train_labels.shape == (
    EXPECTED_TRAIN_PATIENTS,
    NUM_CLASSES
)

assert fusion_val_labels.shape == (
    EXPECTED_VAL_PATIENTS,
    NUM_CLASSES
)


# ============================================================
# 3. VERIFY LABEL VALUES
# ============================================================

assert bool(
    (
        (fusion_train_labels == 0)
        |
        (fusion_train_labels == 1)
    ).all()
)

assert bool(
    (
        (fusion_val_labels == 0)
        |
        (fusion_val_labels == 1)
    ).all()
)


# ============================================================
# 4. VERIFY MULTI-LABEL TARGET DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("MULTI-LABEL TARGET DISTRIBUTION")
print("=" * 80)

print("\nTraining:")

for index, disease in enumerate(
    LABEL_COLUMNS
):

    positive_count = int(
        fusion_train_labels[
            :,
            index
        ].sum().item()
    )

    prevalence = (
        positive_count
        /
        EXPECTED_TRAIN_PATIENTS
    )

    print(
        f"  {disease}: "
        f"{positive_count:4d} positive "
        f"({prevalence:.4f})"
    )


print("\nValidation:")

for index, disease in enumerate(
    LABEL_COLUMNS
):

    positive_count = int(
        fusion_val_labels[
            :,
            index
        ].sum().item()
    )

    prevalence = (
        positive_count
        /
        EXPECTED_VAL_PATIENTS
    )

    print(
        f"  {disease}: "
        f"{positive_count:4d} positive "
        f"({prevalence:.4f})"
    )


# ============================================================
# 5. CREATE MEAN-FUSION BASELINE
# ============================================================

mean_train_fused = (
    fusion_train_memory[
        "right_cross_eye"
    ]
    +
    fusion_train_memory[
        "left_cross_eye"
    ]
) / 2.0


mean_val_fused = (
    fusion_val_memory[
        "right_cross_eye"
    ]
    +
    fusion_val_memory[
        "left_cross_eye"
    ]
) / 2.0


# ============================================================
# 6. VERIFY MEAN FUSION
# ============================================================

assert mean_train_fused.shape == (
    EXPECTED_TRAIN_PATIENTS,
    FEATURE_DIM
)

assert mean_val_fused.shape == (
    EXPECTED_VAL_PATIENTS,
    FEATURE_DIM
)

assert torch.isfinite(
    mean_train_fused
).all()

assert torch.isfinite(
    mean_val_fused
).all()


# ============================================================
# 7. DEFINE QUALITY-ONLY FUSION MODEL
# ============================================================

class QualityOnlyBilateralFusion(
    nn.Module
):

    def __init__(
        self,
        hidden_dim=32
    ):

        super().__init__()

        self.right_gate = nn.Sequential(

            nn.Linear(
                1,
                hidden_dim
            ),

            nn.LayerNorm(
                hidden_dim
            ),

            nn.GELU(),

            nn.Linear(
                hidden_dim,
                1
            )
        )

        self.left_gate = nn.Sequential(

            nn.Linear(
                1,
                hidden_dim
            ),

            nn.LayerNorm(
                hidden_dim
            ),

            nn.GELU(),

            nn.Linear(
                hidden_dim,
                1
            )
        )


    def forward(
        self,
        right_cross_eye,
        left_cross_eye,
        right_quality,
        left_quality
    ):

        if right_quality.dim() == 1:

            right_quality = (
                right_quality.unsqueeze(1)
            )

        if left_quality.dim() == 1:

            left_quality = (
                left_quality.unsqueeze(1)
            )


        right_logit = self.right_gate(
            right_quality
        )

        left_logit = self.left_gate(
            left_quality
        )


        logits = torch.cat(
            [
                right_logit,
                left_logit
            ],
            dim=1
        )


        weights = F.softmax(
            logits,
            dim=1
        )


        right_weight = (
            weights[:, 0:1]
        )

        left_weight = (
            weights[:, 1:2]
        )


        fused = (
            right_weight
            *
            right_cross_eye
        ) + (
            left_weight
            *
            left_cross_eye
        )


        return {
            "fused_features": fused,
            "right_weight": right_weight,
            "left_weight": left_weight,
            "right_logit": right_logit,
            "left_logit": left_logit
        }


# ============================================================
# 8. INSTANTIATE QUALITY-ONLY MODEL
# ============================================================

quality_only_fusion_model = (
    QualityOnlyBilateralFusion(
        hidden_dim=FUSION_HIDDEN_DIM
    )
).to(device)


# ============================================================
# 9. COUNT QUALITY-ONLY PARAMETERS
# ============================================================

quality_only_parameters = sum(
    parameter.numel()
    for parameter
    in quality_only_fusion_model.parameters()
)

print(
    "\nQuality-only fusion parameters:",
    quality_only_parameters
)

assert (
    quality_only_parameters
    ==
    322
)


# ============================================================
# 10. CREATE IDENTICAL PREDICTION HEAD
# ============================================================

class FusionPredictionHead(
    nn.Module
):

    def __init__(
        self,
        feature_dim=768,
        num_classes=8
    ):

        super().__init__()

        self.classifier = nn.Linear(
            feature_dim,
            num_classes
        )


    def forward(
        self,
        features
    ):

        return self.classifier(
            features
        )


# ============================================================
# 11. CREATE HEAD FACTORY
# ============================================================

def create_prediction_head():

    return FusionPredictionHead(
        feature_dim=FEATURE_DIM,
        num_classes=NUM_CLASSES
    ).to(device)


# ============================================================
# 12. VERIFY EXPERIMENT INPUTS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "EXPERIMENT INPUTS"
)

print(
    "=" * 80
)

print(
    "\nExperiment A — Mean Fusion"
)

print(
    "  Right Cross-Eye + Left Cross-Eye"
)

print(
    "  No fusion parameters"
)

print(
    "\nExperiment B — Quality Only"
)

print(
    "  Right quality + Left quality"
)

print(
    "  Trainable eye gates"
)

print(
    "\nExperiment C — Quality + Disease"
)

print(
    "  Right quality + Left quality"
)

print(
    "  Right disease relevance + Left disease relevance"
)

print(
    "  Trainable eye gates"
)


# ============================================================
# 13. VERIFY TRAIN/VALIDATION SEPARATION
# ============================================================

assert (
    len(
        fusion_train_memory[
            "patient_ids"
        ]
    )
    ==
    EXPECTED_TRAIN_PATIENTS
)

assert (
    len(
        fusion_val_memory[
            "patient_ids"
        ]
    )
    ==
    EXPECTED_VAL_PATIENTS
)

assert set(
    fusion_train_memory[
        "patient_ids"
    ]
).isdisjoint(
    set(
        fusion_val_memory[
            "patient_ids"
        ]
    )
)


# ============================================================
# 14. DEFINE EXPERIMENT CONFIGURATION
# ============================================================

FUSION_EXPERIMENTS = {

    "mean":
    {
        "description":
            "Simple equal bilateral averaging",

        "uses_quality":
            False,

        "uses_disease_relevance":
            False,

        "trainable":
            False
    },

    "quality_only":
    {
        "description":
            "Adaptive fusion using image quality",

        "uses_quality":
            True,

        "uses_disease_relevance":
            False,

        "trainable":
            True
    },

    "quality_disease":
    {
        "description":
            "Adaptive fusion using quality and disease relevance",

        "uses_quality":
            True,

        "uses_disease_relevance":
            True,

        "trainable":
            True
    }
}


# ============================================================
# 15. FINAL PROTOCOL SUMMARY
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "CELL 5 — EXPERIMENT PROTOCOL READY"
)

print(
    "=" * 80
)

print(
    "\nTraining patients:",
    EXPECTED_TRAIN_PATIENTS
)

print(
    "Validation patients:",
    EXPECTED_VAL_PATIENTS
)

print(
    "Input representation dimension:",
    FEATURE_DIM
)

print(
    "Output classes:",
    NUM_CLASSES
)

print(
    "\nExperiments:"
)

for name, config in (
    FUSION_EXPERIMENTS.items()
):

    print(
        f"\n  {name}"
    )

    print(
        f"    {config['description']}"
    )

    print(
        f"    Quality: "
        f"{config['uses_quality']}"
    )

    print(
        f"    Disease relevance: "
        f"{config['uses_disease_relevance']}"
    )

    print(
        f"    Trainable: "
        f"{config['trainable']}"
    )


print(
    "\n✓ Target labels loaded"
)

print(
    "✓ Mean-fusion baseline constructed"
)

print(
    "✓ Quality-only fusion defined"
)

print(
    "✓ Common prediction-head architecture defined"
)

print(
    "✓ Experiment configurations registered"
)

print(
    "✓ Train/validation separation verified"
)

print(
    "\nNo experiment training performed."
)

print(
    "Ready for Cell 6: controlled training."
)

ADAPTIVE BILATERAL FUSION — EXPERIMENT PROTOCOL

MULTI-LABEL TARGET DISTRIBUTION

Training:
  N:  730 positive (0.3410)
  D:  719 positive (0.3358)
  G:  131 positive (0.0612)
  C:  139 positive (0.0649)
  A:  105 positive (0.0490)
  H:   70 positive (0.0327)
  M:   99 positive (0.0462)
  O:  474 positive (0.2214)

Validation:
  N:  135 positive (0.3082)
  D:  147 positive (0.3356)
  G:   36 positive (0.0822)
  C:   21 positive (0.0479)
  A:   20 positive (0.0457)
  H:   18 positive (0.0411)
  M:   23 positive (0.0525)
  O:  103 positive (0.2352)

Quality-only fusion parameters: 322

EXPERIMENT INPUTS

Experiment A — Mean Fusion
  Right Cross-Eye + Left Cross-Eye
  No fusion parameters

Experiment B — Quality Only
  Right quality + Left quality
  Trainable eye gates

Experiment C — Quality + Disease
  Right quality + Left quality
  Right disease relevance + Left disease relevance
  Trainable eye gates

CELL 5 — EXPERIMENT PROTOCOL READY

Training patients: 2141
Validation patients: 438


CELL 6 — CONTROLLED ADAPTIVE FUSION TRAINING


This cell performs controlled training and evaluation of the
Adaptive Bilateral Fusion experiments.

All Stage-1 Disease-Aware features and Stage-2 Cross-Eye
representations remain frozen.

Three controlled experiments are evaluated:

    EXPERIMENT A — MEAN FUSION

        F_fused = (F_right + F_left) / 2

        No adaptive fusion parameters are trained.

    EXPERIMENT B — QUALITY-ONLY ADAPTIVE FUSION

        Eye contribution weights are learned using:

            • Right-eye quality
            • Left-eye quality

    EXPERIMENT C — QUALITY + DISEASE ADAPTIVE FUSION

        Eye contribution weights are learned using:

            • Right-eye quality
            • Left-eye quality
            • Right-eye disease relevance
            • Left-eye disease relevance

For the three experiments, the downstream prediction head uses the
same architecture and training protocol.

Training controls:

    • Same random seed
    • Same batch size
    • Same optimizer
    • Same learning rate
    • Same weight decay
    • Same class-weighted BCE loss
    • Same maximum epochs
    • Same early-stopping criterion
    • Same training patients
    • Same validation patients

The 8 disease labels are treated as a multi-label classification
problem.

Class weighting is calculated from the training set to account for
disease imbalance.

Validation data is used only for model selection and final
evaluation.

The Adaptive Fusion module learns normalized bilateral weights:

    w_R + w_L = 1

The final patient representation is:

    F_fused =
        w_R × F_right
        +
        w_L × F_left

For the mean-fusion baseline:

    w_R = 0.5
    w_L = 0.5

The experiment records:

    • Training loss
    • Validation loss
    • Validation Micro-F1
    • Validation Macro-F1
    • Exact-match accuracy
    • Learned right-eye weights
    • Learned left-eye weights

The purpose is to determine:

    1. Whether adaptive bilateral fusion improves over equal
       bilateral averaging.

    2. Whether image quality provides useful information for
       adaptive eye weighting.

    3. Whether the stored disease-relevance signal provides
       additional benefit beyond image quality.

No Cross-Eye representation is fine-tuned during this experiment.

In [9]:
# ============================================================
# CELL 6 — CONTROLLED ADAPTIVE FUSION TRAINING
# ============================================================

import copy

print("=" * 80)
print("ADAPTIVE BILATERAL FUSION — CONTROLLED TRAINING")
print("=" * 80)


# ============================================================
# 1. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# 2. TRAINING CONFIGURATION
# ============================================================

FUSION_BATCH_SIZE = 64
FUSION_EPOCHS = 40
FUSION_LEARNING_RATE = 1e-3
FUSION_WEIGHT_DECAY = 1e-4
FUSION_PATIENCE = 7
FUSION_MIN_DELTA = 1e-4


print("\nTraining configuration:")
print("  Seed:", SEED)
print("  Batch size:", FUSION_BATCH_SIZE)
print("  Maximum epochs:", FUSION_EPOCHS)
print("  Learning rate:", FUSION_LEARNING_RATE)
print("  Weight decay:", FUSION_WEIGHT_DECAY)
print("  Early stopping patience:", FUSION_PATIENCE)


# ============================================================
# 3. LOAD FROZEN MEMORY TO GPU
# ============================================================

train_right_cross = (
    fusion_train_memory[
        "right_cross_eye"
    ].float().to(device)
)

train_left_cross = (
    fusion_train_memory[
        "left_cross_eye"
    ].float().to(device)
)

train_right_quality_device = (
    fusion_train_memory[
        "right_quality"
    ].float().to(device)
)

train_left_quality_device = (
    fusion_train_memory[
        "left_quality"
    ].float().to(device)
)

train_right_relevance_device = (
    fusion_train_memory[
        "right_disease_relevance"
    ].float().to(device)
)

train_left_relevance_device = (
    fusion_train_memory[
        "left_disease_relevance"
    ].float().to(device)
)

train_labels_device = (
    fusion_train_labels.float().to(device)
)


val_right_cross = (
    fusion_val_memory[
        "right_cross_eye"
    ].float().to(device)
)

val_left_cross = (
    fusion_val_memory[
        "left_cross_eye"
    ].float().to(device)
)

val_right_quality_device = (
    fusion_val_memory[
        "right_quality"
    ].float().to(device)
)

val_left_quality_device = (
    fusion_val_memory[
        "left_quality"
    ].float().to(device)
)

val_right_relevance_device = (
    fusion_val_memory[
        "right_disease_relevance"
    ].float().to(device)
)

val_left_relevance_device = (
    fusion_val_memory[
        "left_disease_relevance"
    ].float().to(device)
)

val_labels_device = (
    fusion_val_labels.float().to(device)
)


# ============================================================
# 4. VERIFY GPU TENSORS
# ============================================================

assert train_right_cross.is_cuda
assert train_left_cross.is_cuda
assert train_right_quality_device.is_cuda
assert train_left_quality_device.is_cuda
assert train_right_relevance_device.is_cuda
assert train_left_relevance_device.is_cuda
assert train_labels_device.is_cuda

assert val_right_cross.is_cuda
assert val_left_cross.is_cuda
assert val_right_quality_device.is_cuda
assert val_left_quality_device.is_cuda
assert val_right_relevance_device.is_cuda
assert val_left_relevance_device.is_cuda
assert val_labels_device.is_cuda


print("\n✓ Training tensors on:", train_right_cross.device)
print("✓ Validation tensors on:", val_right_cross.device)


# ============================================================
# 5. VERIFY DATASET SIZES
# ============================================================

assert train_right_cross.shape == (
    EXPECTED_TRAIN_PATIENTS,
    FEATURE_DIM
)

assert train_left_cross.shape == (
    EXPECTED_TRAIN_PATIENTS,
    FEATURE_DIM
)

assert val_right_cross.shape == (
    EXPECTED_VAL_PATIENTS,
    FEATURE_DIM
)

assert val_left_cross.shape == (
    EXPECTED_VAL_PATIENTS,
    FEATURE_DIM
)

assert train_labels_device.shape == (
    EXPECTED_TRAIN_PATIENTS,
    NUM_CLASSES
)

assert val_labels_device.shape == (
    EXPECTED_VAL_PATIENTS,
    NUM_CLASSES
)


# ============================================================
# 6. CLASS-WEIGHTED BCE
# ============================================================

positive_counts = (
    train_labels_device.sum(
        dim=0
    )
)

negative_counts = (
    train_labels_device.shape[0]
    -
    positive_counts
)

pos_weight = (
    negative_counts
    /
    positive_counts.clamp(
        min=1
    )
)


print("\nPositive class weights:")

for disease, weight in zip(
    LABEL_COLUMNS,
    pos_weight.detach().cpu().numpy()
):

    print(
        f"  {disease}: {weight:.4f}"
    )


# ============================================================
# 7. GPU MINI-BATCH GENERATOR
# ============================================================

def generate_gpu_batches(
    num_samples,
    batch_size,
    shuffle=True
):

    if shuffle:

        indices = torch.randperm(
            num_samples,
            device=device
        )

    else:

        indices = torch.arange(
            num_samples,
            device=device
        )


    for start in range(
        0,
        num_samples,
        batch_size
    ):

        end = min(
            start + batch_size,
            num_samples
        )

        yield indices[
            start:end
        ]


# ============================================================
# 8. METRIC FUNCTIONS
# ============================================================

def multilabel_micro_f1(
    logits,
    labels,
    threshold=0.5
):

    probabilities = torch.sigmoid(
        logits
    )

    predictions = (
        probabilities >= threshold
    ).float()

    true_positive = (
        predictions * labels
    ).sum().item()

    false_positive = (
        predictions * (1 - labels)
    ).sum().item()

    false_negative = (
        (1 - predictions) * labels
    ).sum().item()

    denominator = (
        2 * true_positive
        +
        false_positive
        +
        false_negative
    )

    if denominator == 0:

        return 0.0

    return (
        2 * true_positive
        /
        denominator
    )


def multilabel_macro_f1(
    logits,
    labels,
    threshold=0.5
):

    probabilities = torch.sigmoid(
        logits
    )

    predictions = (
        probabilities >= threshold
    ).float()

    f1_scores = []

    for class_index in range(
        labels.shape[1]
    ):

        pred = predictions[
            :,
            class_index
        ]

        true = labels[
            :,
            class_index
        ]

        tp = (
            pred * true
        ).sum().item()

        fp = (
            pred * (1 - true)
        ).sum().item()

        fn = (
            (1 - pred) * true
        ).sum().item()

        denominator = (
            2 * tp
            +
            fp
            +
            fn
        )

        if denominator == 0:

            f1 = 0.0

        else:

            f1 = (
                2 * tp
                /
                denominator
            )

        f1_scores.append(
            f1
        )

    return float(
        np.mean(
            f1_scores
        )
    )


def exact_match_accuracy(
    logits,
    labels,
    threshold=0.5
):

    probabilities = torch.sigmoid(
        logits
    )

    predictions = (
        probabilities >= threshold
    ).float()

    matches = (
        predictions == labels
    ).all(
        dim=1
    )

    return (
        matches.float()
        .mean()
        .item()
    )


# ============================================================
# 9. GENERIC FUSION FORWARD
# ============================================================

def get_fused_features(
    model_type,
    model,
    right_cross,
    left_cross,
    right_quality,
    left_quality,
    right_relevance,
    left_relevance
):

    if model_type == "mean":

        return (
            right_cross
            +
            left_cross
        ) / 2.0


    elif model_type == "quality_only":

        output = model(
            right_cross,
            left_cross,
            right_quality,
            left_quality
        )

        return output[
            "fused_features"
        ]


    elif model_type == "quality_disease":

        output = model(
            right_cross,
            left_cross,
            right_quality,
            left_quality,
            right_relevance,
            left_relevance
        )

        return output[
            "fused_features"
        ]


    else:

        raise ValueError(
            f"Unknown model type: {model_type}"
        )


# ============================================================
# 10. TRAIN ONE EXPERIMENT
# ============================================================

def train_fusion_experiment(
    model_type,
    seed=42
):

    print(
        f"\nInitializing {model_type}..."
    )


    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)


    # --------------------------------------------------------
    # Create fusion model
    # --------------------------------------------------------

    if model_type == "mean":

        fusion_model = None

    elif model_type == "quality_only":

        fusion_model = (
            QualityOnlyBilateralFusion(
                hidden_dim=FUSION_HIDDEN_DIM
            ).to(device)
        )

    elif model_type == "quality_disease":

        fusion_model = (
            AdaptiveBilateralFusion(
                feature_dim=FEATURE_DIM,
                disease_dim=NUM_CLASSES,
                hidden_dim=FUSION_HIDDEN_DIM
            ).to(device)
        )

    else:

        raise ValueError(
            model_type
        )


    # --------------------------------------------------------
    # Create identical prediction head
    # --------------------------------------------------------

    prediction_head = (
        create_prediction_head()
    )


    # --------------------------------------------------------
    # Collect trainable parameters
    # --------------------------------------------------------

    parameters = list(
        prediction_head.parameters()
    )

    if fusion_model is not None:

        parameters.extend(
            list(
                fusion_model.parameters()
            )
        )


    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        parameters,
        lr=FUSION_LEARNING_RATE,
        weight_decay=FUSION_WEIGHT_DECAY
    )


    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )


    # --------------------------------------------------------
    # Best model tracking
    # --------------------------------------------------------

    best_val_loss = float("inf")

    best_epoch = 0

    best_fusion_state = None

    best_head_state = None

    patience_counter = 0

    history = []


    # ========================================================
    # EPOCH LOOP
    # ========================================================

    for epoch in range(
        1,
        FUSION_EPOCHS + 1
    ):


        # ====================================================
        # TRAIN
        # ====================================================

        if fusion_model is not None:

            fusion_model.train()

        prediction_head.train()


        train_loss_sum = 0.0
        train_count = 0

        train_logits_all = []
        train_labels_all = []


        # ----------------------------------------------------
        # GPU mini-batches
        # ----------------------------------------------------

        for batch_indices in generate_gpu_batches(
            EXPECTED_TRAIN_PATIENTS,
            FUSION_BATCH_SIZE,
            shuffle=True
        ):

            optimizer.zero_grad()


            # ------------------------------------------------
            # Gather GPU batch
            # ------------------------------------------------

            right_cross_batch = (
                train_right_cross[
                    batch_indices
                ]
            )

            left_cross_batch = (
                train_left_cross[
                    batch_indices
                ]
            )

            right_quality_batch = (
                train_right_quality_device[
                    batch_indices
                ]
            )

            left_quality_batch = (
                train_left_quality_device[
                    batch_indices
                ]
            )

            right_relevance_batch = (
                train_right_relevance_device[
                    batch_indices
                ]
            )

            left_relevance_batch = (
                train_left_relevance_device[
                    batch_indices
                ]
            )

            labels_batch = (
                train_labels_device[
                    batch_indices
                ]
            )


            # ------------------------------------------------
            # Fusion
            # ------------------------------------------------

            fused_features = (
                get_fused_features(
                    model_type,
                    fusion_model,
                    right_cross_batch,
                    left_cross_batch,
                    right_quality_batch,
                    left_quality_batch,
                    right_relevance_batch,
                    left_relevance_batch
                )
            )


            # ------------------------------------------------
            # Prediction
            # ------------------------------------------------

            logits = prediction_head(
                fused_features
            )


            # ------------------------------------------------
            # Loss
            # ------------------------------------------------

            loss = criterion(
                logits,
                labels_batch
            )


            # ------------------------------------------------
            # Backpropagation
            # ------------------------------------------------

            loss.backward()

            optimizer.step()


            # ------------------------------------------------
            # Statistics
            # ------------------------------------------------

            current_batch_size = (
                labels_batch.shape[0]
            )

            train_loss_sum += (
                loss.item()
                *
                current_batch_size
            )

            train_count += (
                current_batch_size
            )


            train_logits_all.append(
                logits.detach()
            )

            train_labels_all.append(
                labels_batch.detach()
            )


        # ====================================================
        # TRAIN METRICS
        # ====================================================

        train_loss = (
            train_loss_sum
            /
            train_count
        )


        train_logits_cat = torch.cat(
            train_logits_all,
            dim=0
        )

        train_labels_cat = torch.cat(
            train_labels_all,
            dim=0
        )


        train_micro_f1 = (
            multilabel_micro_f1(
                train_logits_cat,
                train_labels_cat
            )
        )

        train_macro_f1 = (
            multilabel_macro_f1(
                train_logits_cat,
                train_labels_cat
            )
        )


        # ====================================================
        # VALIDATION
        # ====================================================

        if fusion_model is not None:

            fusion_model.eval()

        prediction_head.eval()


        val_loss_sum = 0.0
        val_count = 0

        val_logits_all = []
        val_labels_all = []


        with torch.no_grad():

            for batch_indices in generate_gpu_batches(
                EXPECTED_VAL_PATIENTS,
                FUSION_BATCH_SIZE,
                shuffle=False
            ):

                right_cross_batch = (
                    val_right_cross[
                        batch_indices
                    ]
                )

                left_cross_batch = (
                    val_left_cross[
                        batch_indices
                    ]
                )

                right_quality_batch = (
                    val_right_quality_device[
                        batch_indices
                    ]
                )

                left_quality_batch = (
                    val_left_quality_device[
                        batch_indices
                    ]
                )

                right_relevance_batch = (
                    val_right_relevance_device[
                        batch_indices
                    ]
                )

                left_relevance_batch = (
                    val_left_relevance_device[
                        batch_indices
                    ]
                )

                labels_batch = (
                    val_labels_device[
                        batch_indices
                    ]
                )


                # --------------------------------------------
                # Fusion
                # --------------------------------------------

                fused_features = (
                    get_fused_features(
                        model_type,
                        fusion_model,
                        right_cross_batch,
                        left_cross_batch,
                        right_quality_batch,
                        left_quality_batch,
                        right_relevance_batch,
                        left_relevance_batch
                    )
                )


                # --------------------------------------------
                # Prediction
                # --------------------------------------------

                logits = prediction_head(
                    fused_features
                )


                # --------------------------------------------
                # Loss
                # --------------------------------------------

                loss = criterion(
                    logits,
                    labels_batch
                )


                current_batch_size = (
                    labels_batch.shape[0]
                )

                val_loss_sum += (
                    loss.item()
                    *
                    current_batch_size
                )

                val_count += (
                    current_batch_size
                )


                val_logits_all.append(
                    logits.detach()
                )

                val_labels_all.append(
                    labels_batch.detach()
                )


        # ====================================================
        # VALIDATION METRICS
        # ====================================================

        val_loss = (
            val_loss_sum
            /
            val_count
        )


        val_logits_cat = torch.cat(
            val_logits_all,
            dim=0
        )

        val_labels_cat = torch.cat(
            val_labels_all,
            dim=0
        )


        val_micro_f1 = (
            multilabel_micro_f1(
                val_logits_cat,
                val_labels_cat
            )
        )

        val_macro_f1 = (
            multilabel_macro_f1(
                val_logits_cat,
                val_labels_cat
            )
        )

        val_exact_match = (
            exact_match_accuracy(
                val_logits_cat,
                val_labels_cat
            )
        )


        # ====================================================
        # RECORD HISTORY
        # ====================================================

        history.append({

            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "val_loss":
                val_loss,

            "train_micro_f1":
                train_micro_f1,

            "train_macro_f1":
                train_macro_f1,

            "val_micro_f1":
                val_micro_f1,

            "val_macro_f1":
                val_macro_f1,

            "val_exact_match":
                val_exact_match
        })


        # ====================================================
        # PROGRESS
        # ====================================================

        print(
            f"{model_type:16s} "
            f"Epoch {epoch:02d} | "
            f"Train Loss {train_loss:.4f} | "
            f"Val Loss {val_loss:.4f} | "
            f"Val Micro-F1 {val_micro_f1:.4f} | "
            f"Val Macro-F1 {val_macro_f1:.4f}"
        )


        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if (
            val_loss
            <
            best_val_loss
            -
            FUSION_MIN_DELTA
        ):

            best_val_loss = val_loss

            best_epoch = epoch

            patience_counter = 0


            best_head_state = (
                copy.deepcopy(
                    prediction_head.state_dict()
                )
            )


            if fusion_model is not None:

                best_fusion_state = (
                    copy.deepcopy(
                        fusion_model.state_dict()
                    )
                )

        else:

            patience_counter += 1


        if (
            patience_counter
            >=
            FUSION_PATIENCE
        ):

            print(
                f"  Early stopping at epoch {epoch}."
            )

            break


    # ========================================================
    # RESTORE BEST MODEL
    # ========================================================

    prediction_head.load_state_dict(
        best_head_state
    )

    if fusion_model is not None:

        fusion_model.load_state_dict(
            best_fusion_state
        )


    # ========================================================
    # FINAL VALIDATION
    # ========================================================

    if fusion_model is not None:

        fusion_model.eval()

    prediction_head.eval()


    final_logits = []
    final_labels = []

    final_right_weights = []
    final_left_weights = []


    with torch.no_grad():

        for batch_indices in generate_gpu_batches(
            EXPECTED_VAL_PATIENTS,
            FUSION_BATCH_SIZE,
            shuffle=False
        ):

            right_cross_batch = (
                val_right_cross[
                    batch_indices
                ]
            )

            left_cross_batch = (
                val_left_cross[
                    batch_indices
                ]
            )

            right_quality_batch = (
                val_right_quality_device[
                    batch_indices
                ]
            )

            left_quality_batch = (
                val_left_quality_device[
                    batch_indices
                ]
            )

            right_relevance_batch = (
                val_right_relevance_device[
                    batch_indices
                ]
            )

            left_relevance_batch = (
                val_left_relevance_device[
                    batch_indices
                ]
            )

            labels_batch = (
                val_labels_device[
                    batch_indices
                ]
            )


            # ------------------------------------------------
            # Mean
            # ------------------------------------------------

            if model_type == "mean":

                fused = (
                    right_cross_batch
                    +
                    left_cross_batch
                ) / 2.0


            # ------------------------------------------------
            # Quality only
            # ------------------------------------------------

            elif model_type == "quality_only":

                output = fusion_model(
                    right_cross_batch,
                    left_cross_batch,
                    right_quality_batch,
                    left_quality_batch
                )

                fused = output[
                    "fused_features"
                ]

                final_right_weights.append(
                    output[
                        "right_weight"
                    ].detach()
                )

                final_left_weights.append(
                    output[
                        "left_weight"
                    ].detach()
                )


            # ------------------------------------------------
            # Quality + disease
            # ------------------------------------------------

            elif model_type == "quality_disease":

                output = fusion_model(
                    right_cross_batch,
                    left_cross_batch,
                    right_quality_batch,
                    left_quality_batch,
                    right_relevance_batch,
                    left_relevance_batch
                )

                fused = output[
                    "fused_features"
                ]

                final_right_weights.append(
                    output[
                        "right_weight"
                    ].detach()
                )

                final_left_weights.append(
                    output[
                        "left_weight"
                    ].detach()
                )


            else:

                raise ValueError(
                    model_type
                )


            # ------------------------------------------------
            # Prediction
            # ------------------------------------------------

            logits = prediction_head(
                fused
            )


            final_logits.append(
                logits.detach()
            )

            final_labels.append(
                labels_batch.detach()
            )


    final_logits = torch.cat(
        final_logits,
        dim=0
    )

    final_labels = torch.cat(
        final_labels,
        dim=0
    )


    # ========================================================
    # FINAL METRICS
    # ========================================================

    final_micro_f1 = (
        multilabel_micro_f1(
            final_logits,
            final_labels
        )
    )

    final_macro_f1 = (
        multilabel_macro_f1(
            final_logits,
            final_labels
        )
    )

    final_exact_match = (
        exact_match_accuracy(
            final_logits,
            final_labels
        )
    )


    # ========================================================
    # WEIGHT STATISTICS
    # ========================================================

    if model_type != "mean":

        right_weights = torch.cat(
            final_right_weights,
            dim=0
        ).squeeze(1)

        left_weights = torch.cat(
            final_left_weights,
            dim=0
        ).squeeze(1)


        weight_statistics = {

            "right_mean":
                right_weights.mean().item(),

            "right_std":
                right_weights.std().item(),

            "right_min":
                right_weights.min().item(),

            "right_max":
                right_weights.max().item(),

            "left_mean":
                left_weights.mean().item(),

            "left_std":
                left_weights.std().item(),

            "left_min":
                left_weights.min().item(),

            "left_max":
                left_weights.max().item()
        }


    else:

        right_weights = torch.full(
            (EXPECTED_VAL_PATIENTS,),
            0.5
        )

        left_weights = torch.full(
            (EXPECTED_VAL_PATIENTS,),
            0.5
        )


        weight_statistics = {

            "right_mean": 0.5,
            "right_std": 0.0,
            "right_min": 0.5,
            "right_max": 0.5,

            "left_mean": 0.5,
            "left_std": 0.0,
            "left_min": 0.5,
            "left_max": 0.5
        }


    # ========================================================
    # RETURN RESULT
    # ========================================================

    return {

        "model_type":
            model_type,

        "best_epoch":
            best_epoch,

        "best_val_loss":
            best_val_loss,

        "final_val_micro_f1":
            final_micro_f1,

        "final_val_macro_f1":
            final_macro_f1,

        "final_val_exact_match":
            final_exact_match,

        "history":
            history,

        "fusion_model":
            fusion_model,

        "prediction_head":
            prediction_head,

        "val_logits":
            final_logits.cpu(),

        "val_labels":
            final_labels.cpu(),

        "right_weights":
            right_weights.cpu(),

        "left_weights":
            left_weights.cpu(),

        "weight_statistics":
            weight_statistics
    }


# ============================================================
# 11. RUN ALL THREE EXPERIMENTS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "STARTING CONTROLLED EXPERIMENTS"
)

print(
    "=" * 80
)


fusion_results = {}


# ============================================================
# EXPERIMENT A — MEAN FUSION
# ============================================================

print(
    "\n\n"
    + "=" * 80
)

print(
    "EXPERIMENT A — MEAN FUSION"
)

print(
    "=" * 80
)

fusion_results["mean"] = (
    train_fusion_experiment(
        "mean",
        seed=SEED
    )
)


# ============================================================
# EXPERIMENT B — QUALITY ONLY
# ============================================================

print(
    "\n\n"
    + "=" * 80
)

print(
    "EXPERIMENT B — QUALITY ONLY"
)

print(
    "=" * 80
)

fusion_results["quality_only"] = (
    train_fusion_experiment(
        "quality_only",
        seed=SEED
    )
)


# ============================================================
# EXPERIMENT C — QUALITY + DISEASE
# ============================================================

print(
    "\n\n"
    + "=" * 80
)

print(
    "EXPERIMENT C — QUALITY + DISEASE"
)

print(
    "=" * 80
)

fusion_results["quality_disease"] = (
    train_fusion_experiment(
        "quality_disease",
        seed=SEED
    )
)


# ============================================================
# 12. RESULTS TABLE
# ============================================================

print(
    "\n\n"
    + "=" * 80
)

print(
    "CONTROLLED FUSION RESULTS"
)

print(
    "=" * 80
)


results_rows = []

for name, result in (
    fusion_results.items()
):

    results_rows.append({

        "Experiment":
            name,

        "Best Epoch":
            result[
                "best_epoch"
            ],

        "Best Val Loss":
            result[
                "best_val_loss"
            ],

        "Val Micro-F1":
            result[
                "final_val_micro_f1"
            ],

        "Val Macro-F1":
            result[
                "final_val_macro_f1"
            ],

        "Exact Match":
            result[
                "final_val_exact_match"
            ]
    })


results_df = pd.DataFrame(
    results_rows
)

display(
    results_df
)


# ============================================================
# 13. ADAPTIVE WEIGHT STATISTICS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "ADAPTIVE WEIGHT STATISTICS"
)

print(
    "=" * 80
)


for name in [
    "quality_only",
    "quality_disease"
]:

    stats = fusion_results[
        name
    ][
        "weight_statistics"
    ]

    print(
        f"\n{name}"
    )

    print(
        f"  Right mean: "
        f"{stats['right_mean']:.4f}"
    )

    print(
        f"  Right std:  "
        f"{stats['right_std']:.4f}"
    )

    print(
        f"  Right range:"
        f" [{stats['right_min']:.4f}, "
        f"{stats['right_max']:.4f}]"
    )

    print(
        f"  Left mean:  "
        f"{stats['left_mean']:.4f}"
    )

    print(
        f"  Left std:   "
        f"{stats['left_std']:.4f}"
    )

    print(
        f"  Left range:"
        f" [{stats['left_min']:.4f}, "
        f"{stats['left_max']:.4f}]"
    )


# ============================================================
# 14. VERIFY RESULTS
# ============================================================

for name, result in (
    fusion_results.items()
):

    assert np.isfinite(
        result[
            "final_val_micro_f1"
        ]
    )

    assert np.isfinite(
        result[
            "final_val_macro_f1"
        ]
    )

    assert np.isfinite(
        result[
            "best_val_loss"
        ]
    )


# ============================================================
# 15. SAVE EXPERIMENT RESULTS
# ============================================================

FUSION_RESULTS_PATH = os.path.join(
    ROOT,
    "stage2_adaptive_fusion_experiment_results.pt"
)


torch.save(
    {
        name: {
            key: value
            for key, value in result.items()
            if key not in [
                "fusion_model",
                "prediction_head"
            ]
        }
        for name, result
        in fusion_results.items()
    },
    FUSION_RESULTS_PATH
)


# ============================================================
# 16. FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "CELL 6 — CONTROLLED TRAINING COMPLETE"
)

print(
    "=" * 80
)

print(
    "\n✓ Mean fusion evaluated"
)

print(
    "✓ Quality-only adaptive fusion trained"
)

print(
    "✓ Quality + disease adaptive fusion trained"
)

print(
    "✓ Same prediction-head architecture used"
)

print(
    "✓ Same loss and optimization protocol used"
)

print(
    "✓ Cross-Eye representations remained frozen"
)

print(
    "✓ GPU-resident mini-batch training used"
)

print(
    "✓ No CPU pin-memory transfer used"
)

print(
    "✓ Train/validation separation maintained"
)

print(
    "✓ Adaptive eye weights recorded"
)

print(
    "✓ Validation predictions recorded"
)

print(
    "✓ Experiment results saved:"
)

print(
    FUSION_RESULTS_PATH
)

ADAPTIVE BILATERAL FUSION — CONTROLLED TRAINING

Training configuration:
  Seed: 42
  Batch size: 64
  Maximum epochs: 40
  Learning rate: 0.001
  Weight decay: 0.0001
  Early stopping patience: 7

✓ Training tensors on: cuda:0
✓ Validation tensors on: cuda:0

Positive class weights:
  N: 1.9329
  D: 1.9777
  G: 15.3435
  C: 14.4029
  A: 19.3905
  H: 29.5857
  M: 20.6263
  O: 3.5169

STARTING CONTROLLED EXPERIMENTS


EXPERIMENT A — MEAN FUSION

Initializing mean...
mean             Epoch 01 | Train Loss 0.7750 | Val Loss 0.8111 | Val Micro-F1 0.4900 | Val Macro-F1 0.5040
mean             Epoch 02 | Train Loss 0.5775 | Val Loss 0.8292 | Val Micro-F1 0.5264 | Val Macro-F1 0.5450
mean             Epoch 03 | Train Loss 0.5328 | Val Loss 0.8241 | Val Micro-F1 0.5453 | Val Macro-F1 0.5515
mean             Epoch 04 | Train Loss 0.5004 | Val Loss 0.8204 | Val Micro-F1 0.5477 | Val Macro-F1 0.5566
mean             Epoch 05 | Train Loss 0.4732 | Val Loss 0.8037 | Val Micro-F1 0.5245 | Val Macro-

,Experiment,Best Epoch,Best Val Loss,Val Micro-F1,Val Macro-F1,Exact Match
0,mean,5,0.803673,0.524465,0.545717,0.203196
1,quality_only,5,0.807084,0.529101,0.547021,0.203196
2,quality_disease,5,0.820702,0.521607,0.538667,0.200913



ADAPTIVE WEIGHT STATISTICS

quality_only
  Right mean: 0.5089
  Right std:  0.0025
  Right range: [0.4954, 0.5194]
  Left mean:  0.4911
  Left std:   0.0025
  Left range: [0.4806, 0.5046]

quality_disease
  Right mean: 0.5299
  Right std:  0.0315
  Right range: [0.4296, 0.6268]
  Left mean:  0.4701
  Left std:   0.0315
  Left range: [0.3732, 0.5704]

CELL 6 — CONTROLLED TRAINING COMPLETE

✓ Mean fusion evaluated
✓ Quality-only adaptive fusion trained
✓ Quality + disease adaptive fusion trained
✓ Same prediction-head architecture used
✓ Same loss and optimization protocol used
✓ Cross-Eye representations remained frozen
✓ GPU-resident mini-batch training used
✓ No CPU pin-memory transfer used
✓ Train/validation separation maintained
✓ Adaptive eye weights recorded
✓ Validation predictions recorded
✓ Experiment results saved:
/content/drive/My Drive/Eye Disease/Dataset/stage2_adaptive_fusion_experiment_results.pt



ADAPTIVE BILATERAL FUSION — EXPERIMENT D

CLASS-WISE QUALITY + DISEASE ADAPTIVE FUSION


Experiment D introduces disease-specific bilateral fusion while preserving
the stability of the established mean-fusion representation.

Architecture:

  Inputs:
    • Right Cross-Eye representation: (768,)
    • Left Cross-Eye representation:  (768,)
    • Right image quality:             (1,)
    • Left image quality:              (1,)
    • Right DAA disease relevance:     (8,)
    • Left DAA disease relevance:      (8,)

  Class-wise adaptive gating:
    • One bilateral gate for each of the 8 disease classes
    • Each gate predicts an independent right-eye contribution
    • Left-eye contribution is complementary

  For disease class c:

    w_R,c = sigmoid(g_c(q_R, q_L, a_R, a_L))
    w_L,c = 1 - w_R,c

    F_c = w_R,c F_R + w_L,c F_L

  This produces 8 disease-specific fused representations:
    (8, 768)

Residual mean-fusion anchor:

    F_mean = (F_R + F_L) / 2

    F_final,c =
        F_mean + alpha_c (F_c - F_mean)

  The class-wise residual scales alpha_c are initialized to zero.

Therefore, at initialization:

    F_final,c = F_mean

  and Experiment D begins from the stable symmetric bilateral baseline.

Classification:

  • Eight independent binary classification heads
  • One head per disease class
  • Each head receives its corresponding (768,) class representation

Training protocol:

  Seeds: 42, 123, 456, 789
  Batch size: 64
  Maximum epochs: 40
  Learning rate: 0.001
  Weight decay: 0.0001
  Early stopping patience: 7
  Cross-Eye representations: frozen

The purpose of Experiment D is to determine whether disease-specific
right/left fusion provides useful information beyond symmetric mean fusion
while retaining the stability of the baseline.

No A/B/C models are retrained in this cell.
Experiment D is trained independently and its results are saved for the
evaluation performed in Cell 8.




In [10]:
# ============================================================
# CELL 7 — EXPERIMENT D
# CLASS-WISE QUALITY + DISEASE ADAPTIVE FUSION
# ============================================================

import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score


print("=" * 80)
print("ADAPTIVE BILATERAL FUSION — EXPERIMENT D")
print("CLASS-WISE QUALITY + DISEASE ADAPTIVE FUSION")
print("=" * 80)


# ============================================================
# 1. CONTROLLED CONFIGURATION
# ============================================================

D_SEEDS = [42, 123, 456, 789]

D_BATCH_SIZE = 64
D_MAX_EPOCHS = 40
D_LEARNING_RATE = 0.001
D_WEIGHT_DECAY = 0.0001
D_PATIENCE = 7

D_NUM_CLASSES = 8
D_FEATURE_DIM = 768
D_HIDDEN_DIM = 16

print("\nTraining configuration:")
print("  Seeds:", D_SEEDS)
print("  Batch size:", D_BATCH_SIZE)
print("  Maximum epochs:", D_MAX_EPOCHS)
print("  Learning rate:", D_LEARNING_RATE)
print("  Weight decay:", D_WEIGHT_DECAY)
print("  Early stopping patience:", D_PATIENCE)

print("\nArchitecture:")
print("  Class-wise bilateral gates: 8")
print("  Gate input dimension: 18")
print("  Gate hidden dimension:", D_HIDDEN_DIM)
print("  Disease-specific fused representation: (8, 768)")
print("  Residual anchor: mean bilateral representation")
print("  Residual initialization: zero")
print("  Classification heads: 8 independent binary heads")


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

def set_d_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 3. LOAD FROZEN FUSION MEMORY
# ============================================================

print("\n" + "=" * 80)
print("LOADING FROZEN FUSION MEMORY")
print("=" * 80)


TRAIN_MEMORY_PATH = os.path.join(
    ROOT,
    "stage2_train_adaptive_fusion_memory.pt"
)

VAL_MEMORY_PATH = os.path.join(
    ROOT,
    "stage2_val_adaptive_fusion_memory.pt"
)


train_memory = torch.load(
    TRAIN_MEMORY_PATH,
    map_location="cpu",
    weights_only=False
)

val_memory = torch.load(
    VAL_MEMORY_PATH,
    map_location="cpu",
    weights_only=False
)


print("✓ Training memory loaded")
print("✓ Validation memory loaded")


# ============================================================
# 4. EXTRACT TENSORS
# ============================================================

train_right = train_memory["right_cross_eye"].float()
train_left = train_memory["left_cross_eye"].float()

val_right = val_memory["right_cross_eye"].float()
val_left = val_memory["left_cross_eye"].float()


train_right_quality = train_memory["right_quality"].float()
train_left_quality = train_memory["left_quality"].float()

val_right_quality = val_memory["right_quality"].float()
val_left_quality = val_memory["left_quality"].float()


train_right_disease = train_memory[
    "right_disease_relevance"
].float()

train_left_disease = train_memory[
    "left_disease_relevance"
].float()

val_right_disease = val_memory[
    "right_disease_relevance"
].float()

val_left_disease = val_memory[
    "left_disease_relevance"
].float()


print("\nTraining tensors:")
print("  Right:", tuple(train_right.shape))
print("  Left:", tuple(train_left.shape))
print("  Right quality:", tuple(train_right_quality.shape))
print("  Left quality:", tuple(train_left_quality.shape))
print("  Right disease:", tuple(train_right_disease.shape))
print("  Left disease:", tuple(train_left_disease.shape))


print("\nValidation tensors:")
print("  Right:", tuple(val_right.shape))
print("  Left:", tuple(val_left.shape))
print("  Right quality:", tuple(val_right_quality.shape))
print("  Left quality:", tuple(val_left_quality.shape))
print("  Right disease:", tuple(val_right_disease.shape))
print("  Left disease:", tuple(val_left_disease.shape))


# ============================================================
# 5. LOAD PATIENT-LEVEL LABELS
# ============================================================

LABEL_COLUMNS_D = [
    "N", "D", "G", "C",
    "A", "H", "M", "O"
]


TRAIN_DF_PATH = os.path.join(
    ROOT,
    "stage2_train_df.csv"
)

VAL_DF_PATH = os.path.join(
    ROOT,
    "stage2_val_bilateral_metadata.csv"
)


train_df_d = pd.read_csv(TRAIN_DF_PATH)
val_df_d = pd.read_csv(VAL_DF_PATH)


# ------------------------------------------------------------
# Verify expected row counts
# ------------------------------------------------------------

assert len(train_df_d) == len(train_right), (
    f"Training alignment mismatch: "
    f"{len(train_df_d)} labels vs {len(train_right)} features"
)

assert len(val_df_d) == len(val_right), (
    f"Validation alignment mismatch: "
    f"{len(val_df_d)} labels vs {len(val_right)} features"
)


train_labels_d = torch.tensor(
    train_df_d[LABEL_COLUMNS_D].values,
    dtype=torch.float32
)

val_labels_d = torch.tensor(
    val_df_d[LABEL_COLUMNS_D].values,
    dtype=torch.float32
)


print("\n✓ Training labels:", tuple(train_labels_d.shape))
print("✓ Validation labels:", tuple(val_labels_d.shape))


# ============================================================
# 6. VERIFY ALL INPUTS
# ============================================================

assert train_right.shape == (
    len(train_labels_d),
    D_FEATURE_DIM
)

assert train_left.shape == (
    len(train_labels_d),
    D_FEATURE_DIM
)

assert val_right.shape == (
    len(val_labels_d),
    D_FEATURE_DIM
)

assert val_left.shape == (
    len(val_labels_d),
    D_FEATURE_DIM
)

assert train_right_disease.shape == (
    len(train_labels_d),
    D_NUM_CLASSES
)

assert train_left_disease.shape == (
    len(train_labels_d),
    D_NUM_CLASSES
)

assert val_right_disease.shape == (
    len(val_labels_d),
    D_NUM_CLASSES
)

assert val_left_disease.shape == (
    len(val_labels_d),
    D_NUM_CLASSES
)


assert torch.isfinite(train_right).all()
assert torch.isfinite(train_left).all()
assert torch.isfinite(val_right).all()
assert torch.isfinite(val_left).all()

assert torch.isfinite(train_right_quality).all()
assert torch.isfinite(train_left_quality).all()
assert torch.isfinite(val_right_quality).all()
assert torch.isfinite(val_left_quality).all()

assert torch.isfinite(train_right_disease).all()
assert torch.isfinite(train_left_disease).all()
assert torch.isfinite(val_right_disease).all()
assert torch.isfinite(val_left_disease).all()


print("\n✓ All Experiment D source tensors verified")
print("✓ Training/validation alignment verified")
print("✓ All tensors finite")


# ============================================================
# 7. EXPERIMENT D MODEL
# ============================================================

class ClassWiseDiseaseGate(nn.Module):

    def __init__(
        self,
        input_dim=18,
        hidden_dim=16,
        num_classes=8
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim
            ),

            nn.LayerNorm(
                hidden_dim
            ),

            nn.GELU(),

            nn.Linear(
                hidden_dim,
                num_classes
            )
        )


    def forward(self, x):

        # Independent disease-specific right-eye weights
        right_weights = torch.sigmoid(
            self.network(x)
        )

        # Complementary left-eye weights
        left_weights = 1.0 - right_weights

        return (
            right_weights,
            left_weights
        )


class ExperimentDModel(nn.Module):

    def __init__(
        self,
        feature_dim=768,
        num_classes=8,
        hidden_dim=16
    ):

        super().__init__()

        self.num_classes = num_classes
        self.feature_dim = feature_dim


        # ----------------------------------------------------
        # Disease-specific bilateral gate
        #
        # Input:
        #   q_R, q_L, disease_R[8], disease_L[8]
        #
        # Total = 18
        # ----------------------------------------------------

        self.gate = ClassWiseDiseaseGate(
            input_dim=18,
            hidden_dim=hidden_dim,
            num_classes=num_classes
        )


        # ----------------------------------------------------
        # Zero-initialized class-wise residual scales
        # ----------------------------------------------------

        self.alpha = nn.Parameter(
            torch.zeros(num_classes)
        )


        # ----------------------------------------------------
        # Eight independent binary classifiers
        # ----------------------------------------------------

        self.class_heads = nn.ModuleList([

            nn.Linear(
                feature_dim,
                1
            )

            for _ in range(num_classes)

        ])


    def forward(
        self,
        right_features,
        left_features,
        right_quality,
        left_quality,
        right_disease,
        left_disease
    ):

        batch_size = right_features.shape[0]


        # ----------------------------------------------------
        # Prepare gate input
        # ----------------------------------------------------

        gate_input = torch.cat(
            [
                right_quality.unsqueeze(1),
                left_quality.unsqueeze(1),
                right_disease,
                left_disease
            ],
            dim=1
        )


        # ----------------------------------------------------
        # Class-wise bilateral weights
        # ----------------------------------------------------

        right_weights, left_weights = self.gate(
            gate_input
        )


        # ----------------------------------------------------
        # Stable mean-fusion anchor
        # ----------------------------------------------------

        mean_features = (
            right_features +
            left_features
        ) / 2.0


        # ----------------------------------------------------
        # Disease-specific fused representations
        #
        # Shape:
        #   [B, 8, 768]
        # ----------------------------------------------------

        classwise_features = (

            right_weights.unsqueeze(-1)
            * right_features.unsqueeze(1)

            +

            left_weights.unsqueeze(-1)
            * left_features.unsqueeze(1)
        )


        # ----------------------------------------------------
        # Residual adaptation around mean fusion
        #
        # alpha starts at zero.
        #
        # Therefore initially:
        #
        #   final = mean
        # ----------------------------------------------------

        alpha = self.alpha.view(
            1,
            self.num_classes,
            1
        )


        final_class_features = (

            mean_features.unsqueeze(1)

            +

            alpha
            * (
                classwise_features
                -
                mean_features.unsqueeze(1)
            )
        )


        # ----------------------------------------------------
        # Independent disease classification heads
        # ----------------------------------------------------

        logits = []

        for c in range(
            self.num_classes
        ):

            class_feature = (
                final_class_features[:, c, :]
            )

            class_logit = self.class_heads[c](
                class_feature
            ).squeeze(1)

            logits.append(
                class_logit
            )


        logits = torch.stack(
            logits,
            dim=1
        )


        return {

            "logits":
                logits,

            "right_weights":
                right_weights,

            "left_weights":
                left_weights,

            "classwise_features":
                classwise_features,

            "final_class_features":
                final_class_features,

            "mean_features":
                mean_features,

            "alpha":
                self.alpha
        }


# ============================================================
# 8. MODEL SANITY CHECK
# ============================================================

print("\n" + "=" * 80)
print("EXPERIMENT D MODEL SANITY CHECK")
print("=" * 80)


set_d_seed(42)


device_d = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


test_model_d = ExperimentDModel(
    feature_dim=D_FEATURE_DIM,
    num_classes=D_NUM_CLASSES,
    hidden_dim=D_HIDDEN_DIM
).to(device_d)


print(test_model_d)


trainable_params_d = sum(
    p.numel()
    for p in test_model_d.parameters()
    if p.requires_grad
)


print(
    "\nTrainable parameters:",
    trainable_params_d
)


# ------------------------------------------------------------
# Zero-alpha verification
# ------------------------------------------------------------

with torch.no_grad():

    test_alpha = test_model_d.alpha.detach().cpu()

    print(
        "Initial alpha:",
        test_alpha
    )

    assert torch.allclose(
        test_alpha,
        torch.zeros_like(test_alpha)
    )


print("✓ Residual scales initialized exactly at zero")


# ============================================================
# 9. DATASET
# ============================================================

class ExperimentDDataset(Dataset):

    def __init__(
        self,
        right_features,
        left_features,
        right_quality,
        left_quality,
        right_disease,
        left_disease,
        labels
    ):

        self.right_features = right_features
        self.left_features = left_features

        self.right_quality = right_quality
        self.left_quality = left_quality

        self.right_disease = right_disease
        self.left_disease = left_disease

        self.labels = labels


    def __len__(self):

        return len(
            self.labels
        )


    def __getitem__(self, idx):

        return {

            "right_features":
                self.right_features[idx],

            "left_features":
                self.left_features[idx],

            "right_quality":
                self.right_quality[idx],

            "left_quality":
                self.left_quality[idx],

            "right_disease":
                self.right_disease[idx],

            "left_disease":
                self.left_disease[idx],

            "labels":
                self.labels[idx]
        }


# ============================================================
# 10. BUILD DATASETS
# ============================================================

train_dataset_d = ExperimentDDataset(

    train_right,
    train_left,

    train_right_quality,
    train_left_quality,

    train_right_disease,
    train_left_disease,

    train_labels_d
)


val_dataset_d = ExperimentDDataset(

    val_right,
    val_left,

    val_right_quality,
    val_left_quality,

    val_right_disease,
    val_left_disease,

    val_labels_d
)


# ============================================================
# 11. POSITIVE CLASS WEIGHTS
# ============================================================

positive_counts_d = train_labels_d.sum(
    dim=0
)

negative_counts_d = (
    len(train_labels_d)
    -
    positive_counts_d
)


pos_weight_d = (
    negative_counts_d /
    positive_counts_d.clamp_min(1.0)
)


print("\nPositive class weights:")

for i, label in enumerate(
    LABEL_COLUMNS_D
):

    print(
        f"  {label}: "
        f"{pos_weight_d[i].item():.4f}"
    )


# ============================================================
# 12. SINGLE-SEED TRAINING FUNCTION
# ============================================================

def train_experiment_d(seed):

    set_d_seed(seed)


    train_generator = torch.Generator()

    train_generator.manual_seed(seed)


    train_loader = DataLoader(

        train_dataset_d,

        batch_size=D_BATCH_SIZE,

        shuffle=True,

        generator=train_generator,

        num_workers=0,

        pin_memory=False
    )


    val_loader = DataLoader(

        val_dataset_d,

        batch_size=D_BATCH_SIZE,

        shuffle=False,

        num_workers=0,

        pin_memory=False
    )


    model = ExperimentDModel(

        feature_dim=D_FEATURE_DIM,

        num_classes=D_NUM_CLASSES,

        hidden_dim=D_HIDDEN_DIM

    ).to(device_d)


    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight_d.to(device_d)
    )


    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=D_LEARNING_RATE,

        weight_decay=D_WEIGHT_DECAY
    )


    best_val_loss = float("inf")

    best_epoch = 0

    best_state = None

    best_predictions = None

    best_targets = None

    best_right_weights = None

    best_left_weights = None


    patience_counter = 0


    print("\n" + "-" * 80)
    print(f"EXPERIMENT D | SEED {seed}")
    print("-" * 80)


    for epoch in range(
        1,
        D_MAX_EPOCHS + 1
    ):

        # ====================================================
        # TRAIN
        # ====================================================

        model.train()

        train_loss_sum = 0.0

        train_samples = 0


        for batch in train_loader:

            right_features = batch[
                "right_features"
            ].to(
                device_d,
                non_blocking=True
            )

            left_features = batch[
                "left_features"
            ].to(
                device_d,
                non_blocking=True
            )

            right_quality = batch[
                "right_quality"
            ].to(
                device_d,
                non_blocking=True
            )

            left_quality = batch[
                "left_quality"
            ].to(
                device_d,
                non_blocking=True
            )

            right_disease = batch[
                "right_disease"
            ].to(
                device_d,
                non_blocking=True
            )

            left_disease = batch[
                "left_disease"
            ].to(
                device_d,
                non_blocking=True
            )

            labels = batch[
                "labels"
            ].to(
                device_d,
                non_blocking=True
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            output = model(

                right_features,
                left_features,

                right_quality,
                left_quality,

                right_disease,
                left_disease
            )


            loss = criterion(
                output["logits"],
                labels
            )


            loss.backward()

            optimizer.step()


            batch_size = labels.shape[0]

            train_loss_sum += (
                loss.item()
                *
                batch_size
            )

            train_samples += batch_size


        train_loss = (
            train_loss_sum /
            train_samples
        )


        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()


        val_loss_sum = 0.0

        val_samples = 0


        all_probabilities = []

        all_targets = []

        all_right_weights = []

        all_left_weights = []


        with torch.no_grad():

            for batch in val_loader:

                right_features = batch[
                    "right_features"
                ].to(
                    device_d,
                    non_blocking=True
                )

                left_features = batch[
                    "left_features"
                ].to(
                    device_d,
                    non_blocking=True
                )

                right_quality = batch[
                    "right_quality"
                ].to(
                    device_d,
                    non_blocking=True
                )

                left_quality = batch[
                    "left_quality"
                ].to(
                    device_d,
                    non_blocking=True
                )

                right_disease = batch[
                    "right_disease"
                ].to(
                    device_d,
                    non_blocking=True
                )

                left_disease = batch[
                    "left_disease"
                ].to(
                    device_d,
                    non_blocking=True
                )

                labels = batch[
                    "labels"
                ].to(
                    device_d,
                    non_blocking=True
                )


                output = model(

                    right_features,
                    left_features,

                    right_quality,
                    left_quality,

                    right_disease,
                    left_disease
                )


                loss = criterion(
                    output["logits"],
                    labels
                )


                batch_size = labels.shape[0]

                val_loss_sum += (
                    loss.item()
                    *
                    batch_size
                )

                val_samples += batch_size


                probabilities = torch.sigmoid(
                    output["logits"]
                )


                all_probabilities.append(
                    probabilities.cpu()
                )

                all_targets.append(
                    labels.cpu()
                )

                all_right_weights.append(
                    output[
                        "right_weights"
                    ].cpu()
                )

                all_left_weights.append(
                    output[
                        "left_weights"
                    ].cpu()
                )


        val_loss = (
            val_loss_sum /
            val_samples
        )


        probabilities = torch.cat(
            all_probabilities,
            dim=0
        ).numpy()


        targets = torch.cat(
            all_targets,
            dim=0
        ).numpy()


        right_weights = torch.cat(
            all_right_weights,
            dim=0
        ).numpy()


        left_weights = torch.cat(
            all_left_weights,
            dim=0
        ).numpy()


        predictions = (
            probabilities >= 0.5
        ).astype(np.int32)


        micro_f1 = f1_score(

            targets.reshape(-1),

            predictions.reshape(-1),

            zero_division=0
        )


        macro_f1 = f1_score(

            targets,

            predictions,

            average="macro",

            zero_division=0
        )


        exact_match = np.mean(
            np.all(
                predictions == targets,
                axis=1
            )
        )


        print(

            f"Experiment D | "
            f"Epoch {epoch:02d} | "
            f"Train Loss {train_loss:.4f} | "
            f"Val Loss {val_loss:.4f} | "
            f"Val Micro-F1 {micro_f1:.4f} | "
            f"Val Macro-F1 {macro_f1:.4f}"
        )


        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_epoch = epoch

            patience_counter = 0


            best_state = {

                key:
                    value.detach().cpu().clone()

                for key, value
                in model.state_dict().items()
            }


            best_predictions = predictions.copy()

            best_targets = targets.copy()

            best_right_weights = right_weights.copy()

            best_left_weights = left_weights.copy()


        else:

            patience_counter += 1


        if patience_counter >= D_PATIENCE:

            print(
                f"  Early stopping at epoch {epoch}."
            )

            break


    # ========================================================
    # RESTORE BEST MODEL
    # ========================================================

    model.load_state_dict(
        best_state
    )

    model.to(device_d)

    model.eval()


    # ========================================================
    # FINAL BEST-EPOCH METRICS
    # ========================================================

    final_micro_f1 = f1_score(

        best_targets.reshape(-1),

        best_predictions.reshape(-1),

        zero_division=0
    )


    final_macro_f1 = f1_score(

        best_targets,

        best_predictions,

        average="macro",

        zero_division=0
    )


    final_exact_match = np.mean(

        np.all(
            best_predictions ==
            best_targets,
            axis=1
        )
    )


    # ========================================================
    # CLASS-WISE WEIGHT STATISTICS
    # ========================================================

    right_mean_per_class = (
        best_right_weights.mean(axis=0)
    )

    right_std_per_class = (
        best_right_weights.std(axis=0)
    )

    right_min_per_class = (
        best_right_weights.min(axis=0)
    )

    right_max_per_class = (
        best_right_weights.max(axis=0)
    )


    left_mean_per_class = (
        best_left_weights.mean(axis=0)
    )

    left_std_per_class = (
        best_left_weights.std(axis=0)
    )

    left_min_per_class = (
        best_left_weights.min(axis=0)
    )

    left_max_per_class = (
        best_left_weights.max(axis=0)
    )


    alpha_values = (
        model.alpha.detach()
        .cpu()
        .numpy()
    )


    weight_statistics = {

        "right_mean_per_class":
            right_mean_per_class,

        "right_std_per_class":
            right_std_per_class,

        "right_min_per_class":
            right_min_per_class,

        "right_max_per_class":
            right_max_per_class,

        "left_mean_per_class":
            left_mean_per_class,

        "left_std_per_class":
            left_std_per_class,

        "left_min_per_class":
            left_min_per_class,

        "left_max_per_class":
            left_max_per_class,

        "alpha":
            alpha_values
    }


    return {

        "seed":
            seed,

        "best_epoch":
            best_epoch,

        "best_val_loss":
            best_val_loss,

        "final_val_micro_f1":
            final_micro_f1,

        "final_val_macro_f1":
            final_macro_f1,

        "final_val_exact_match":
            final_exact_match,

        "predictions":
            best_predictions,

        "targets":
            best_targets,

        "right_weights":
            best_right_weights,

        "left_weights":
            best_left_weights,

        "weight_statistics":
            weight_statistics,

        "model_state_dict":
            best_state
    }


# ============================================================
# 13. RUN EXPERIMENT D — FOUR SEEDS
# ============================================================

print("\n" + "=" * 80)
print("STARTING EXPERIMENT D — FOUR-SEED TRAINING")
print("=" * 80)


experiment_d_results = {}


for seed in D_SEEDS:

    experiment_d_results[
        seed
    ] = train_experiment_d(
        seed
    )


# ============================================================
# 14. BUILD EXPERIMENT D SUMMARY
# ============================================================

d_summary_rows = []


for seed in D_SEEDS:

    result = experiment_d_results[
        seed
    ]

    d_summary_rows.append({

        "Seed":
            seed,

        "Best Epoch":
            result[
                "best_epoch"
            ],

        "Best Val Loss":
            result[
                "best_val_loss"
            ],

        "Val Micro-F1":
            result[
                "final_val_micro_f1"
            ],

        "Val Macro-F1":
            result[
                "final_val_macro_f1"
            ],

        "Exact Match":
            result[
                "final_val_exact_match"
            ]
    })


experiment_d_summary_df = pd.DataFrame(
    d_summary_rows
)


print("\n" + "=" * 80)
print("EXPERIMENT D — FOUR-SEED RESULTS")
print("=" * 80)

display(
    experiment_d_summary_df
)


# ============================================================
# 15. MEAN ± STD
# ============================================================

d_mean_std = {

    "Best Val Loss Mean":
        experiment_d_summary_df[
            "Best Val Loss"
        ].mean(),

    "Best Val Loss Std":
        experiment_d_summary_df[
            "Best Val Loss"
        ].std(),

    "Micro-F1 Mean":
        experiment_d_summary_df[
            "Val Micro-F1"
        ].mean(),

    "Micro-F1 Std":
        experiment_d_summary_df[
            "Val Micro-F1"
        ].std(),

    "Macro-F1 Mean":
        experiment_d_summary_df[
            "Val Macro-F1"
        ].mean(),

    "Macro-F1 Std":
        experiment_d_summary_df[
            "Val Macro-F1"
        ].std(),

    "Exact Match Mean":
        experiment_d_summary_df[
            "Exact Match"
        ].mean(),

    "Exact Match Std":
        experiment_d_summary_df[
            "Exact Match"
        ].std()
}


print("\n" + "=" * 80)
print("EXPERIMENT D — MEAN ± STD")
print("=" * 80)


print(
    f"Best Val Loss : "
    f"{d_mean_std['Best Val Loss Mean']:.4f} ± "
    f"{d_mean_std['Best Val Loss Std']:.4f}"
)

print(
    f"Micro-F1      : "
    f"{d_mean_std['Micro-F1 Mean']:.4f} ± "
    f"{d_mean_std['Micro-F1 Std']:.4f}"
)

print(
    f"Macro-F1      : "
    f"{d_mean_std['Macro-F1 Mean']:.4f} ± "
    f"{d_mean_std['Macro-F1 Std']:.4f}"
)

print(
    f"Exact Match   : "
    f"{d_mean_std['Exact Match Mean']:.4f} ± "
    f"{d_mean_std['Exact Match Std']:.4f}"
)


# ============================================================
# 16. CLASS-WISE WEIGHT SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("EXPERIMENT D — CLASS-WISE RIGHT-EYE WEIGHTS")
print("=" * 80)


class_weight_rows = []


for seed in D_SEEDS:

    stats = experiment_d_results[
        seed
    ][
        "weight_statistics"
    ]


    for c, label in enumerate(
        LABEL_COLUMNS_D
    ):

        class_weight_rows.append({

            "Seed":
                seed,

            "Disease":
                label,

            "Right Mean":
                stats[
                    "right_mean_per_class"
                ][c],

            "Right Std":
                stats[
                    "right_std_per_class"
                ][c],

            "Right Min":
                stats[
                    "right_min_per_class"
                ][c],

            "Right Max":
                stats[
                    "right_max_per_class"
                ][c],

            "Left Mean":
                stats[
                    "left_mean_per_class"
                ][c],

            "Alpha":
                stats[
                    "alpha"
                ][c]
        })


d_class_weights_df = pd.DataFrame(
    class_weight_rows
)


display(
    d_class_weights_df
)


# ============================================================
# 17. SAVE EXPERIMENT D RESULTS
# ============================================================

EXPERIMENT_D_RESULTS_PATH = os.path.join(
    ROOT,
    "stage2_experiment_d_classwise_results.pt"
)


torch.save(

    {

        "experiment":
            "D",

        "description":
            "Class-wise Quality + Disease Adaptive Fusion",

        "seeds":
            D_SEEDS,

        "configuration":
            {

                "batch_size":
                    D_BATCH_SIZE,

                "max_epochs":
                    D_MAX_EPOCHS,

                "learning_rate":
                    D_LEARNING_RATE,

                "weight_decay":
                    D_WEIGHT_DECAY,

                "patience":
                    D_PATIENCE,

                "feature_dim":
                    D_FEATURE_DIM,

                "num_classes":
                    D_NUM_CLASSES,

                "hidden_dim":
                    D_HIDDEN_DIM
            },

        "label_columns":
            LABEL_COLUMNS_D,

        "per_seed_results":
            experiment_d_results,

        "summary":
            experiment_d_summary_df,

        "mean_std":
            d_mean_std,

        "classwise_weights":
            d_class_weights_df
    },

    EXPERIMENT_D_RESULTS_PATH
)


print("\n" + "=" * 80)
print("CELL 7 — EXPERIMENT D COMPLETE")
print("=" * 80)

print(
    "\n✓ Class-wise disease-specific gates defined"
)

print(
    "✓ Quality and DAA disease relevance used"
)

print(
    "✓ Independent bilateral weighting for all 8 diseases"
)

print(
    "✓ Mean-fusion residual anchor included"
)

print(
    "✓ Residual scales initialized at zero"
)

print(
    "✓ Eight independent disease classification heads used"
)

print(
    "✓ Cross-Eye representations remained frozen"
)

print(
    "✓ Four random seeds evaluated:",
    D_SEEDS
)

print(
    "✓ Validation predictions recorded"
)

print(
    "✓ Class-wise fusion weights recorded"
)

print(
    "✓ Experiment D results saved:"
)

print(
    EXPERIMENT_D_RESULTS_PATH
)

ADAPTIVE BILATERAL FUSION — EXPERIMENT D
CLASS-WISE QUALITY + DISEASE ADAPTIVE FUSION

Training configuration:
  Seeds: [42, 123, 456, 789]
  Batch size: 64
  Maximum epochs: 40
  Learning rate: 0.001
  Weight decay: 0.0001
  Early stopping patience: 7

Architecture:
  Class-wise bilateral gates: 8
  Gate input dimension: 18
  Gate hidden dimension: 16
  Disease-specific fused representation: (8, 768)
  Residual anchor: mean bilateral representation
  Residual initialization: zero
  Classification heads: 8 independent binary heads

LOADING FROZEN FUSION MEMORY
✓ Training memory loaded
✓ Validation memory loaded

Training tensors:
  Right: (2141, 768)
  Left: (2141, 768)
  Right quality: (2141,)
  Left quality: (2141,)
  Right disease: (2141, 8)
  Left disease: (2141, 8)

Validation tensors:
  Right: (438, 768)
  Left: (438, 768)
  Right quality: (438,)
  Left quality: (438,)
  Right disease: (438, 8)
  Left disease: (438, 8)

✓ Training labels: (2141, 8)
✓ Validation labels: (438, 8)



,Seed,Best Epoch,Best Val Loss,Val Micro-F1,Val Macro-F1,Exact Match
0,42,5,0.791177,0.528541,0.555427,0.175799
1,123,4,0.793274,0.554869,0.559034,0.216895
2,456,4,0.793804,0.532164,0.550781,0.187215
3,789,2,0.811629,0.504302,0.536516,0.118721



EXPERIMENT D — MEAN ± STD
Best Val Loss : 0.7975 ± 0.0095
Micro-F1      : 0.5300 ± 0.0207
Macro-F1      : 0.5504 ± 0.0099
Exact Match   : 0.1747 ± 0.0411

EXPERIMENT D — CLASS-WISE RIGHT-EYE WEIGHTS


,Seed,Disease,Right Mean,Right Std,Right Min,Right Max,Left Mean,Alpha
0,42,N,0.392493,0.012598,0.364112,0.443973,0.607507,-0.007416
1,42,D,0.656096,0.017237,0.545137,0.700744,0.343904,-0.053534
2,42,G,0.457411,0.021961,0.410916,0.529544,0.542589,-0.030679
3,42,C,0.559589,0.009736,0.529296,0.592951,0.440411,0.005342
4,42,A,0.378198,0.011241,0.331479,0.405142,0.621802,0.021231
5,42,H,0.492823,0.017677,0.452531,0.552856,0.507177,-0.002584
6,42,M,0.315352,0.008261,0.286959,0.349968,0.684648,-0.051778
7,42,O,0.314854,0.011550,0.277533,0.367044,0.685146,-0.020915
8,123,N,0.624807,0.019282,0.549390,0.648974,0.375193,0.010495
9,123,D,0.460521,0.019762,0.437877,0.593736,0.539479,0.021709



CELL 7 — EXPERIMENT D COMPLETE

✓ Class-wise disease-specific gates defined
✓ Quality and DAA disease relevance used
✓ Independent bilateral weighting for all 8 diseases
✓ Mean-fusion residual anchor included
✓ Residual scales initialized at zero
✓ Eight independent disease classification heads used
✓ Cross-Eye representations remained frozen
✓ Four random seeds evaluated: [42, 123, 456, 789]
✓ Validation predictions recorded
✓ Class-wise fusion weights recorded
✓ Experiment D results saved:
/content/drive/My Drive/Eye Disease/Dataset/stage2_experiment_d_classwise_results.pt



EXPERIMENT E — STRICT CLASS-WISE ADAPTIVE BILATERAL FUSION

Experiment E tests whether disease-specific bilateral routing can provide
additional information beyond symmetric mean fusion without using a residual
mean-fusion escape path.

Architecture:
  • Frozen Right and Left Cross-Eye representations
  • Quality + disease relevance as gate inputs
  • Class-wise bilateral weights for all 8 diseases
  • Independent sigmoid gating for each disease
  • Temperature-scaled gating
  • No residual mean-fusion anchor
  • No flattening or averaging of class-wise representations
  • Independent binary classification head for each disease

For each disease c:

  w_R,c = sigmoid(z_c / T)
  w_L,c = 1 - w_R,c

  F_c = w_R,c F_R + w_L,c F_L

The resulting representation has shape:

  [batch, 8, 768]

Each disease-specific representation is passed to its corresponding
binary classification head.

The purpose of Experiment E is to directly test whether quality- and
disease-aware class-wise routing can learn useful eye-specific contributions
when the model is not given a residual pathway back to mean fusion.

Experiment D remains unchanged and serves as the residual-anchored
class-wise adaptive fusion comparison.

No cross-eye representations are retrained.
The training/validation split and optimization protocol remain controlled.

In [12]:
# ============================================================
# CELL 8 — EXPERIMENT E
# STRICT CLASS-WISE ADAPTIVE BILATERAL FUSION
# ============================================================

print("=" * 80)
print("EXPERIMENT E — STRICT CLASS-WISE ADAPTIVE BILATERAL FUSION")
print("=" * 80)


# ============================================================
# 1. CONFIGURATION
# ============================================================

E_HIDDEN_DIM = 16
E_TEMPERATURE = 1.0

print("\nConfiguration:")
print("  Gate input dimension:", 18)
print("  Hidden dimension:", E_HIDDEN_DIM)
print("  Disease classes:", NUM_CLASSES)
print("  Temperature:", E_TEMPERATURE)
print("  Residual mean anchor: NONE")
print("  Cross-Eye representations: FROZEN")


# ============================================================
# 2. STRICT CLASS-WISE ADAPTIVE GATE
# ============================================================

class StrictClassWiseAdaptiveGate(nn.Module):

    def __init__(
        self,
        input_dim=18,
        hidden_dim=16,
        num_classes=8,
        temperature=1.0
    ):

        super().__init__()

        self.temperature = temperature

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim
            ),

            nn.LayerNorm(
                hidden_dim
            ),

            nn.GELU(),

            nn.Linear(
                hidden_dim,
                num_classes
            )
        )


    def forward(
        self,
        right_quality,
        left_quality,
        right_disease,
        left_disease
    ):

        x = torch.cat(
            [
                right_quality.unsqueeze(1),
                left_quality.unsqueeze(1),
                right_disease,
                left_disease
            ],
            dim=1
        )

        logits = self.network(x)

        right_weights = torch.sigmoid(
            logits / self.temperature
        )

        left_weights = 1.0 - right_weights

        return (
            right_weights,
            left_weights
        )


# ============================================================
# 3. STRICT CLASS-WISE FUSION
# ============================================================

class StrictClassWiseAdaptiveFusion(nn.Module):

    def __init__(
        self,
        feature_dim=768,
        num_classes=8,
        hidden_dim=16,
        temperature=1.0
    ):

        super().__init__()

        self.feature_dim = feature_dim
        self.num_classes = num_classes

        self.gate = StrictClassWiseAdaptiveGate(
            input_dim=2 + 2 * num_classes,
            hidden_dim=hidden_dim,
            num_classes=num_classes,
            temperature=temperature
        )


    def forward(
        self,
        right_features,
        left_features,
        right_quality,
        left_quality,
        right_disease,
        left_disease
    ):

        right_weights, left_weights = self.gate(
            right_quality,
            left_quality,
            right_disease,
            left_disease
        )

        # ----------------------------------------------------
        # Class-wise bilateral fusion
        #
        # right_features: [B, 768]
        # left_features : [B, 768]
        #
        # weights: [B, 8]
        #
        # output: [B, 8, 768]
        # ----------------------------------------------------

        fused = (
            right_weights.unsqueeze(-1)
            * right_features.unsqueeze(1)
            +
            left_weights.unsqueeze(-1)
            * left_features.unsqueeze(1)
        )

        return (
            fused,
            right_weights,
            left_weights
        )


# ============================================================
# 4. CLASS-SPECIFIC PREDICTION HEADS
# ============================================================

class ClassSpecificPredictionHeads(nn.Module):

    def __init__(
        self,
        feature_dim=768,
        num_classes=8
    ):

        super().__init__()

        self.heads = nn.ModuleList(
            [
                nn.Linear(
                    feature_dim,
                    1
                )
                for _ in range(num_classes)
            ]
        )


    def forward(
        self,
        class_features
    ):

        outputs = []

        for c in range(
            self.heads.__len__()
        ):

            class_feature = class_features[
                :, c, :
            ]

            output = self.heads[c](
                class_feature
            )

            outputs.append(
                output
            )

        return torch.cat(
            outputs,
            dim=1
        )


# ============================================================
# 5. BUILD EXPERIMENT E MODEL
# ============================================================

experiment_e_fusion = StrictClassWiseAdaptiveFusion(
    feature_dim=FEATURE_DIM,
    num_classes=NUM_CLASSES,
    hidden_dim=E_HIDDEN_DIM,
    temperature=E_TEMPERATURE
)


experiment_e_head = ClassSpecificPredictionHeads(
    feature_dim=FEATURE_DIM,
    num_classes=NUM_CLASSES
)


experiment_e_fusion = (
    experiment_e_fusion
    .to(device)
)

experiment_e_head = (
    experiment_e_head
    .to(device)
)


fusion_params = sum(
    p.numel()
    for p in experiment_e_fusion.parameters()
    if p.requires_grad
)

head_params = sum(
    p.numel()
    for p in experiment_e_head.parameters()
    if p.requires_grad
)

print("\nModel:")
print(experiment_e_fusion)

print("\nClass-specific prediction heads:")
print(experiment_e_head)

print("\nTrainable parameters:")
print("  Fusion:", fusion_params)
print("  Heads :", head_params)
print("  Total :", fusion_params + head_params)


# ============================================================
# 6. SANITY CHECK
# ============================================================

experiment_e_fusion.eval()
experiment_e_head.eval()

with torch.no_grad():

    sample_n = min(
        4,
        train_right_cross.shape[0]
    )

    sample_fused, sample_wr, sample_wl = (
        experiment_e_fusion(
            train_right_cross[
                :sample_n
            ],

            train_left_cross[
                :sample_n
            ],

            train_right_quality_device[
                :sample_n
            ],

            train_left_quality_device[
                :sample_n
            ],

            train_right_relevance_device[
                :sample_n
            ],

            train_left_relevance_device[
                :sample_n
            ]
        )
    )

    sample_logits = experiment_e_head(
        sample_fused
    )


print("\n" + "=" * 80)
print("EXPERIMENT E SANITY CHECK")
print("=" * 80)

print(
    "Fused class representation:",
    tuple(sample_fused.shape)
)

print(
    "Expected:",
    f"({sample_n}, {NUM_CLASSES}, {FEATURE_DIM})"
)

print(
    "Right class-wise weights:",
    tuple(sample_wr.shape)
)

print(
    "Left class-wise weights:",
    tuple(sample_wl.shape)
)

print(
    "Prediction logits:",
    tuple(sample_logits.shape)
)

print(
    "Weight sum min:",
    (sample_wr + sample_wl).min().item()
)

print(
    "Weight sum max:",
    (sample_wr + sample_wl).max().item()
)

print(
    "Right weight min/max:",
    sample_wr.min().item(),
    sample_wr.max().item()
)

print(
    "Left weight min/max:",
    sample_wl.min().item(),
    sample_wl.max().item()
)

print(
    "All fused features finite:",
    torch.isfinite(sample_fused).all().item()
)

print(
    "All logits finite:",
    torch.isfinite(sample_logits).all().item()
)


# ============================================================
# 7. VERIFY NO RESIDUAL MEAN ANCHOR
# ============================================================

assert sample_fused.shape == (
    sample_n,
    NUM_CLASSES,
    FEATURE_DIM
)

assert sample_wr.shape == (
    sample_n,
    NUM_CLASSES
)

assert sample_wl.shape == (
    sample_n,
    NUM_CLASSES
)

assert sample_logits.shape == (
    sample_n,
    NUM_CLASSES
)

assert torch.allclose(
    sample_wr + sample_wl,
    torch.ones_like(sample_wr),
    atol=1e-6
)

assert torch.isfinite(
    sample_fused
).all()

assert torch.isfinite(
    sample_logits
).all()


print("\n" + "=" * 80)
print("CELL 8 — EXPERIMENT E MODULE READY")
print("=" * 80)

print("\n✓ Class-wise adaptive gate defined")
print("✓ Quality information included")
print("✓ Disease relevance included")
print("✓ 8 independent bilateral weights produced")
print("✓ Sigmoid gating used")
print("✓ Temperature scaling enabled")
print("✓ No residual mean-fusion anchor")
print("✓ Class representations preserved as [B, 8, 768]")
print("✓ Disease-specific prediction heads defined")
print("✓ Cross-Eye representations remain frozen")
print("✓ Tensor dimensions verified")
print("✓ Weight complementarity verified")
print("✓ All outputs finite")

print(
    "\nReady for Experiment E controlled training."
)


EXPERIMENT E — STRICT CLASS-WISE ADAPTIVE BILATERAL FUSION

Configuration:
  Gate input dimension: 18
  Hidden dimension: 16
  Disease classes: 8
  Temperature: 1.0
  Residual mean anchor: NONE
  Cross-Eye representations: FROZEN

Model:
StrictClassWiseAdaptiveFusion(
  (gate): StrictClassWiseAdaptiveGate(
    (network): Sequential(
      (0): Linear(in_features=18, out_features=16, bias=True)
      (1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
      (3): Linear(in_features=16, out_features=8, bias=True)
    )
  )
)

Class-specific prediction heads:
ClassSpecificPredictionHeads(
  (heads): ModuleList(
    (0-7): 8 x Linear(in_features=768, out_features=1, bias=True)
  )
)

Trainable parameters:
  Fusion: 472
  Heads : 6152
  Total : 6624

EXPERIMENT E SANITY CHECK
Fused class representation: (4, 8, 768)
Expected: (4, 8, 768)
Right class-wise weights: (4, 8)
Left class-wise weights: (4, 8)
Prediction logits: (4, 8)
Weight sum min: 1.0
Weig


ADAPTIVE BILATERAL FUSION — FOUR-SEED EXPERIMENT EVALUATION


Evaluation protocol:

  Seeds:
    42
    123
    456
    789

  Experiments:
    A — Mean Fusion
    B — Quality Only
    C — Quality + Disease
    D — Class-Wise Quality + Disease + Residual Anchor

  Purpose:
    Compare all four fusion strategies under identical
    four-seed evaluation conditions.

  Evaluation metrics:
    Best Validation Loss
    Validation Micro-F1
    Validation Macro-F1
    Exact Match

  Additional analysis:
    Adaptive bilateral weight statistics
    Per-experiment mean ± standard deviation
    Comparison against Mean Fusion baseline
    Experiment D improvement/degradation relative to baseline

  No model training performed.
  No new architecture introduced.
  No modification of frozen Stage-1 or Cross-Eye representations.



In [13]:
# ============================================================
# CELL 9 — FOUR-SEED ROBUSTNESS EVALUATION
# ============================================================

print("=" * 80)
print("ADAPTIVE BILATERAL FUSION — FOUR-SEED ROBUSTNESS EVALUATION")
print("=" * 80)


# ============================================================
# 1. ROBUSTNESS CONFIGURATION
# ============================================================

ROBUSTNESS_SEEDS = [42, 123, 456, 789]

ALL_EXPERIMENTS = [
    "mean",
    "quality_only",
    "quality_disease",
    "experiment_d"
]

EXPERIMENT_NAMES = {
    "mean":
        "Mean Fusion",

    "quality_only":
        "Quality Only",

    "quality_disease":
        "Quality + Disease",

    "experiment_d":
        "Experiment D — Class-wise Disease-Aware Residual"
}


print("\nSeeds:")
print(ROBUSTNESS_SEEDS)

print("\nExperiments:")
for exp in ALL_EXPERIMENTS:
    print(
        f"  {exp:<18} "
        f"{EXPERIMENT_NAMES[exp]}"
    )

print("\nProtocol:")
print("  Same training/validation split")
print("  Same frozen Cross-Eye representations")
print("  Same prediction-head design where applicable")
print("  Same loss function")
print("  Same optimizer")
print("  Same batch size")
print("  Same maximum epochs")
print("  Same early-stopping protocol")
print("  Only random seed is changed")

print("\nNo data or backbone retraining introduced.")


# ============================================================
# 2. RUN FOUR-SEED ROBUSTNESS EXPERIMENTS
# ============================================================

multi_seed_results = {}


for seed in ROBUSTNESS_SEEDS:

    print("\n\n" + "=" * 80)
    print(f"SEED {seed}")
    print("=" * 80)

    multi_seed_results[seed] = {}


    for model_type in ALL_EXPERIMENTS:

        print("\n" + "-" * 80)
        print(
            f"{EXPERIMENT_NAMES[model_type]} "
            f"| Seed: {seed}"
        )
        print("-" * 80)


        # ----------------------------------------------------
        # EXISTING EXPERIMENTS A-C
        # ----------------------------------------------------

        if model_type in [
            "mean",
            "quality_only",
            "quality_disease"
        ]:

            result = train_fusion_experiment(
                model_type,
                seed=seed
            )


        # ----------------------------------------------------
        # EXPERIMENT D
        # ----------------------------------------------------
        #
        # Cell 7 must define the Experiment-D trainer as:
        #
        # train_experiment_d(...)
        #
        # It should return the same core result fields:
        # best_epoch
        # best_val_loss
        # final_val_micro_f1
        # final_val_macro_f1
        # final_val_exact_match
        # weight_statistics
        #
        # ----------------------------------------------------

        elif model_type == "experiment_d":

            result = train_experiment_d(
                seed=seed
            )


        else:

            raise ValueError(
                f"Unknown experiment: {model_type}"
            )


        multi_seed_results[
            seed
        ][
            model_type
        ] = result


# ============================================================
# 3. BUILD COMPLETE PER-SEED RESULTS TABLE
# ============================================================

robustness_rows = []


for seed in ROBUSTNESS_SEEDS:

    for model_type in ALL_EXPERIMENTS:

        result = multi_seed_results[
            seed
        ][
            model_type
        ]


        robustness_rows.append({

            "Seed":
                seed,

            "Experiment":
                EXPERIMENT_NAMES[
                    model_type
                ],

            "Best Epoch":
                result[
                    "best_epoch"
                ],

            "Best Val Loss":
                result[
                    "best_val_loss"
                ],

            "Val Micro-F1":
                result[
                    "final_val_micro_f1"
                ],

            "Val Macro-F1":
                result[
                    "final_val_macro_f1"
                ],

            "Exact Match":
                result[
                    "final_val_exact_match"
                ]
        })


robustness_df = pd.DataFrame(
    robustness_rows
)


print("\n\n" + "=" * 80)
print("FOUR-SEED ROBUSTNESS RESULTS")
print("=" * 80)

display(
    robustness_df
)


# ============================================================
# 4. MEAN ± STANDARD DEVIATION
# ============================================================

summary_df = (
    robustness_df
    .groupby(
        "Experiment"
    )[
        [
            "Best Val Loss",
            "Val Micro-F1",
            "Val Macro-F1",
            "Exact Match"
        ]
    ]
    .agg(
        ["mean", "std"]
    )
    .reset_index()
)


print("\n" + "=" * 80)
print("FOUR-SEED MEAN ± STD")
print("=" * 80)

display(
    summary_df
)


# ============================================================
# 5. CLEAN COMPARISON TABLE
# ============================================================

comparison_rows = []


for experiment_name in [
    EXPERIMENT_NAMES["mean"],
    EXPERIMENT_NAMES["quality_only"],
    EXPERIMENT_NAMES["quality_disease"],
    EXPERIMENT_NAMES["experiment_d"]
]:

    row = summary_df[
        summary_df[
            "Experiment"
        ]
        ==
        experiment_name
    ]


    if len(row) == 0:
        continue


    row = row.iloc[0]


    comparison_rows.append({

        "Experiment":
            experiment_name,

        "Val Loss":
            f"{row[('Best Val Loss', 'mean')]:.4f} ± "
            f"{row[('Best Val Loss', 'std')]:.4f}",

        "Micro-F1":
            f"{row[('Val Micro-F1', 'mean')]:.4f} ± "
            f"{row[('Val Micro-F1', 'std')]:.4f}",

        "Macro-F1":
            f"{row[('Val Macro-F1', 'mean')]:.4f} ± "
            f"{row[('Val Macro-F1', 'std')]:.4f}",

        "Exact Match":
            f"{row[('Exact Match', 'mean')]:.4f} ± "
            f"{row[('Exact Match', 'std')]:.4f}"
    })


comparison_df = pd.DataFrame(
    comparison_rows
)


print("\n" + "=" * 80)
print("FINAL FOUR-EXPERIMENT COMPARISON")
print("=" * 80)

display(
    comparison_df
)


# ============================================================
# 6. ADAPTIVE WEIGHT ROBUSTNESS
# ============================================================

weight_rows = []


for seed in ROBUSTNESS_SEEDS:

    for model_type in [
        "quality_only",
        "quality_disease",
        "experiment_d"
    ]:

        result = multi_seed_results[
            seed
        ][
            model_type
        ]


        stats = result.get(
            "weight_statistics",
            None
        )


        if stats is None:
            continue


        weight_row = {

            "Seed":
                seed,

            "Experiment":
                EXPERIMENT_NAMES[
                    model_type
                ]
        }


        # ----------------------------------------------------
        # Scalar-gate experiments
        # ----------------------------------------------------

        if (
            "right_mean" in stats
        ):

            weight_row.update({

                "Right Mean":
                    stats[
                        "right_mean"
                    ],

                "Right Std":
                    stats[
                        "right_std"
                    ],

                "Right Min":
                    stats[
                        "right_min"
                    ],

                "Right Max":
                    stats[
                        "right_max"
                    ],

                "Left Mean":
                    stats[
                        "left_mean"
                    ],

                "Left Std":
                    stats[
                        "left_std"
                    ],

                "Left Min":
                    stats[
                        "left_min"
                    ],

                "Left Max":
                    stats[
                        "left_max"
                    ]
            })


        # ----------------------------------------------------
        # Class-wise Experiment-D statistics
        # ----------------------------------------------------

        else:

            weight_row.update({

                "Right Mean":
                    stats.get(
                        "right_mean",
                        np.nan
                    ),

                "Right Std":
                    stats.get(
                        "right_std",
                        np.nan
                    ),

                "Right Min":
                    stats.get(
                        "right_min",
                        np.nan
                    ),

                "Right Max":
                    stats.get(
                        "right_max",
                        np.nan
                    ),

                "Left Mean":
                    stats.get(
                        "left_mean",
                        np.nan
                    ),

                "Left Std":
                    stats.get(
                        "left_std",
                        np.nan
                    ),

                "Left Min":
                    stats.get(
                        "left_min",
                        np.nan
                    ),

                "Left Max":
                    stats.get(
                        "left_max",
                        np.nan
                    )
            })


        weight_rows.append(
            weight_row
        )


weight_df = pd.DataFrame(
    weight_rows
)


print("\n" + "=" * 80)
print("ADAPTIVE WEIGHT ROBUSTNESS")
print("=" * 80)

display(
    weight_df
)


# ============================================================
# 7. DETERMINE BEST STRATEGY
# ============================================================

print("\n" + "=" * 80)
print("ROBUSTNESS CONCLUSION")
print("=" * 80)


macro_summary = (
    robustness_df
    .groupby(
        "Experiment"
    )[
        "Val Macro-F1"
    ]
    .agg(
        ["mean", "std"]
    )
    .sort_values(
        "mean",
        ascending=False
    )
)


print("\nRanking by mean Macro-F1:")

for rank, (
    experiment,
    values
) in enumerate(
    macro_summary.iterrows(),
    start=1
):

    print(
        f"{rank}. "
        f"{experiment}: "
        f"{values['mean']:.4f} ± "
        f"{values['std']:.4f}"
    )


best_experiment = (
    macro_summary
    .index[0]
)


best_macro_mean = (
    macro_summary.iloc[0]["mean"]
)


print(
    f"\nBest four-seed Macro-F1: "
    f"{best_experiment}"
)

print(
    f"Mean Macro-F1: "
    f"{best_macro_mean:.4f}"
)


# ============================================================
# 8. SAVE COMPLETE ROBUSTNESS RESULTS
# ============================================================

ROBUSTNESS_RESULTS_PATH = os.path.join(
    ROOT,
    "stage2_adaptive_fusion_four_seed_robustness.pt"
)


serializable_results = {}


for seed, seed_results in (
    multi_seed_results.items()
):

    serializable_results[
        seed
    ] = {}


    for model_type, result in (
        seed_results.items()
    ):

        serializable_results[
            seed
        ][
            model_type
        ] = {

            key: value

            for key, value
            in result.items()

            if key not in [
                "fusion_model",
                "prediction_head",
                "model",
                "optimizer"
            ]
        }


torch.save(
    {

        "seeds":
            ROBUSTNESS_SEEDS,

        "experiments":
            ALL_EXPERIMENTS,

        "experiment_names":
            EXPERIMENT_NAMES,

        "results":
            serializable_results,

        "per_seed_table":
            robustness_df,

        "summary":
            summary_df,

        "comparison":
            comparison_df,

        "weight_statistics":
            weight_df,

        "best_experiment":
            best_experiment
    },

    ROBUSTNESS_RESULTS_PATH
)


# ============================================================
# 9. COMPLETION
# ============================================================

print("\n" + "=" * 80)
print("CELL 8 — FOUR-SEED ROBUSTNESS EVALUATION COMPLETE")
print("=" * 80)

print(
    "\n✓ Four seeds evaluated:",
    ROBUSTNESS_SEEDS
)

print(
    "✓ Mean Fusion evaluated"
)

print(
    "✓ Quality-only Fusion evaluated"
)

print(
    "✓ Quality + Disease Fusion evaluated"
)

print(
    "✓ Experiment D evaluated"
)

print(
    "✓ Per-seed validation metrics recorded"
)

print(
    "✓ Mean ± standard deviation calculated"
)

print(
    "✓ Adaptive weight statistics recorded"
)

print(
    "✓ Best strategy identified by mean Macro-F1"
)

print(
    "✓ Complete robustness results saved:"
)

print(
    ROBUSTNESS_RESULTS_PATH
)

ADAPTIVE BILATERAL FUSION — FOUR-SEED ROBUSTNESS EVALUATION

Seeds:
[42, 123, 456, 789]

Experiments:
  mean               Mean Fusion
  quality_only       Quality Only
  quality_disease    Quality + Disease
  experiment_d       Experiment D — Class-wise Disease-Aware Residual

Protocol:
  Same training/validation split
  Same frozen Cross-Eye representations
  Same prediction-head design where applicable
  Same loss function
  Same optimizer
  Same batch size
  Same maximum epochs
  Same early-stopping protocol
  Only random seed is changed

No data or backbone retraining introduced.


SEED 42

--------------------------------------------------------------------------------
Mean Fusion | Seed: 42
--------------------------------------------------------------------------------

Initializing mean...
mean             Epoch 01 | Train Loss 0.7750 | Val Loss 0.8111 | Val Micro-F1 0.4900 | Val Macro-F1 0.5040
mean             Epoch 02 | Train Loss 0.5775 | Val Loss 0.8292 | Val Micro-F1 0.5

,Seed,Experiment,Best Epoch,Best Val Loss,Val Micro-F1,Val Macro-F1,Exact Match
0,42,Mean Fusion,5,0.803673,0.524465,0.545717,0.203196
1,42,Quality Only,5,0.807084,0.529101,0.547021,0.203196
2,42,Quality + Disease,5,0.820702,0.521607,0.538667,0.200913
3,42,Experiment D — Class-wise Disease-Aware Residual,5,0.791177,0.528541,0.555427,0.175799
4,123,Mean Fusion,5,0.792944,0.560490,0.568800,0.228310
5,123,Quality Only,5,0.799040,0.556669,0.561045,0.212329
6,123,Quality + Disease,5,0.802483,0.560790,0.563943,0.230594
7,123,Experiment D — Class-wise Disease-Aware Residual,4,0.793274,0.554869,0.559034,0.216895
8,456,Mean Fusion,4,0.794492,0.534639,0.535735,0.191781
9,456,Quality Only,4,0.784383,0.539683,0.548611,0.196347



FOUR-SEED MEAN ± STD


Experiment Best Val Loss            \
                                                             mean       std   
0  Experiment D — Class-wise Disease-Aware Residual      0.797471  0.009507   
1                                       Mean Fusion      0.791075  0.012828   
2                                 Quality + Disease      0.802060  0.016232   
3                                      Quality Only      0.791324  0.014486   

  Val Micro-F1           Val Macro-F1           Exact Match            
          mean       std         mean       std        mean       std  
0     0.529969  0.020701     0.550440  0.009878    0.174658  0.041117  
1     0.534768  0.018271     0.550872  0.013937    0.187785  0.042769  
2     0.531795  0.020065     0.545212  0.012794    0.188927  0.045942  
3     0.533331  0.020421     0.550141  0.007529    0.185502  0.037486


FINAL FOUR-EXPERIMENT COMPARISON


,Experiment,Val Loss,Micro-F1,Macro-F1,Exact Match
0,Mean Fusion,0.7911 ± 0.0128,0.5348 ± 0.0183,0.5509 ± 0.0139,0.1878 ± 0.0428
1,Quality Only,0.7913 ± 0.0145,0.5333 ± 0.0204,0.5501 ± 0.0075,0.1855 ± 0.0375
2,Quality + Disease,0.8021 ± 0.0162,0.5318 ± 0.0201,0.5452 ± 0.0128,0.1889 ± 0.0459
3,Experiment D — Class-wise Disease-Aware Residual,0.7975 ± 0.0095,0.5300 ± 0.0207,0.5504 ± 0.0099,0.1747 ± 0.0411



ADAPTIVE WEIGHT ROBUSTNESS


,Seed,Experiment,Right Mean,Right Std,Right Min,Right Max,Left Mean,Left Std,Left Min,Left Max
0,42,Quality Only,0.508904,0.002467,0.495441,0.519397,0.491096,0.002467,0.480603,0.504559
1,42,Quality + Disease,0.529887,0.031544,0.429600,0.626826,0.470113,0.031544,0.373174,0.570400
2,42,Experiment D — Class-wise Disease-Aware Residual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,123,Quality Only,0.521959,0.007413,0.484465,0.555700,0.478041,0.007413,0.444300,0.515535
4,123,Quality + Disease,0.493368,0.046520,0.398055,0.679655,0.506632,0.046520,0.320345,0.601946
5,123,Experiment D — Class-wise Disease-Aware Residual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,456,Quality Only,0.480734,0.004942,0.474782,0.515423,0.519266,0.004942,0.484577,0.525218
7,456,Quality + Disease,0.454249,0.039048,0.346555,0.631068,0.545751,0.039048,0.368932,0.653444
8,456,Experiment D — Class-wise Disease-Aware Residual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,789,Quality Only,0.537933,0.006456,0.483624,0.547013,0.462067,0.006456,0.452987,0.516376



ROBUSTNESS CONCLUSION

Ranking by mean Macro-F1:
1. Mean Fusion: 0.5509 ± 0.0139
2. Experiment D — Class-wise Disease-Aware Residual: 0.5504 ± 0.0099
3. Quality Only: 0.5501 ± 0.0075
4. Quality + Disease: 0.5452 ± 0.0128

Best four-seed Macro-F1: Mean Fusion
Mean Macro-F1: 0.5509

CELL 8 — FOUR-SEED ROBUSTNESS EVALUATION COMPLETE

✓ Four seeds evaluated: [42, 123, 456, 789]
✓ Mean Fusion evaluated
✓ Quality-only Fusion evaluated
✓ Quality + Disease Fusion evaluated
✓ Experiment D evaluated
✓ Per-seed validation metrics recorded
✓ Mean ± standard deviation calculated
✓ Adaptive weight statistics recorded
✓ Best strategy identified by mean Macro-F1
✓ Complete robustness results saved:
/content/drive/My Drive/Eye Disease/Dataset/stage2_adaptive_fusion_four_seed_robustness.pt
